# Haru Colab - MKV Muxing & Extract Tool

**Jalur utama (disarankan):** jalankan **1A Setup**, lalu **1B** (install CLI + web terminal otomatis). Ketik `haru-mux` / `haru-extract` / `haru-metadata` di terminal yang terbuka. Semua alur download - edit - mux/extract - upload + notif Telegram ada di terminal.

**Jalur alternatif:** cell form satu-per-satu di bawah (download, register track, edit, mux, mediainfo, upload). Boleh diskip kalau pakai web terminal.
> Butuh downloader YouTube / LRC / MangaDex? Buka `aio.ipynb` (satu repo, pola pakai sama).


---

### Persiapan (sebelum pakai)

Buka menu **Rahasia** (ikon kunci di sidebar kiri), lalu tambah secret berikut (aktifkan toggle akses notebook-nya):

| `GOFILE_API_TOKEN` | `fb` | Token filmbeehub proxy (download/upload Gofile) |
| `GDRIVE_CLIENT_ID` | *(dari Google Cloud Console)* | OAuth Client ID untuk Google Drive |
| `GDRIVE_CLIENT_SECRET` | *(dari Google Cloud Console)* | OAuth Client Secret untuk Google Drive |
| `GDRIVE_REFRESH_TOKEN` | *(dari OAuth flow)* | OAuth Refresh Token untuk Google Drive |
| `OWNER_ID` | *(Telegram chat ID)* | Untuk auto-post link terminal & hasil ke Telegram |
| `HARU_BOT_TOKEN` | *(token BotFather)* | Token bot Telegram khusus Haru (jangan pakai BOT_TOKEN lain) |

> **Google Drive:** Jika sudah punya `GDRIVE_CLIENT_ID`, `GDRIVE_CLIENT_SECRET`, dan `GDRIVE_REFRESH_TOKEN`, cell Google Drive akan otomatis pakai auth tersebut. Jika belum, cukup klik **Hubungkan** saat cell pertama dijalankan (menggunakan auth bawaan Colab).

## 1 — Setup

In [ ]:
!apt-get update -qq && apt-get install -y -qq mkvtoolnix mediainfo > /dev/null 2>&1
!mkvmerge --version

import subprocess, json, os, re, shutil, hashlib, urllib.parse
from pathlib import Path
from typing import Optional
import requests

UPLOAD_DIR = Path('/content/uploads')
UPLOAD_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = Path('/content/output')
OUTPUT_DIR.mkdir(exist_ok=True)

print('✅ Ready')

## 1B — Web Terminal + CLI Tools (cukup 1A + ini)
Install `haru-mux` & `haru-extract` & `haru-metadata` otomatis, lalu buka terminal di browser. Copy-paste & arrow keys jalan.


In [ ]:
#@title Buka Web Terminal + CLI { display-mode: "form" }
import subprocess, os, time, re, requests
from IPython.display import HTML, display

import subprocess, os

print('📦 Install tools...')
subprocess.run(['apt-get', 'update', '-qq'], capture_output=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'mkvtoolnix', 'mediainfo', 'tmux', 'jq', 'tree', 'wget', 'curl'], capture_output=True)

# Install Python deps
subprocess.run(['pip', 'install', '-q', 'gdown'], capture_output=True)

print('📦 Install haru-mux...')
HARU_MUX_SCRIPT = r'''#!/usr/bin/env python3
import subprocess,sys,os,re,glob,json,time
import requests
from pathlib import Path
V={'.mkv','.mp4','.avi','.mov','.webm','.flv','.wmv','.ts','.m4v'}
A={'.mp3','.aac','.flac','.wav','.ogg','.opus','.mka','.ac3','.dts','.eac3','.m4a'}
S={'.srt','.ass','.ssa','.sub','.idx','.sup','.vtt','.pgs','.scc','.sami'}
L={'id':'Indonesian','en':'English','ja':'Japanese','ko':'Korean','zh':'Chinese','ms':'Malay','ar':'Arabic','de':'German','fr':'French','es':'Spanish','pt':'Portuguese','ru':'Russian','it':'Italian','th':'Thai','vi':'Vietnamese','hi':'Hindi','und':'Undetermined'}
UPLOAD=Path('/content/uploads')
OUTPUT=Path('/content/output')
OUTPUT.mkdir(exist_ok=True)
TGBOT=''
def tg_owner():
 return get_secret('OWNER_ID')
def tg_token():
 return get_secret('HARU_BOT_TOKEN')
def tg_send(msg):
 oid=tg_owner()
 tok=tg_token()
 if not oid or not tok:return
 try:requests.post('https://api.telegram.org/bot'+tok+'/sendMessage',json={'chat_id':oid,'text':msg,'parse_mode':'HTML','disable_web_page_preview':True},timeout=10)
 except:pass
def ci():os.system('cls' if os.name=='nt' else 'clear')
def ok(t):return '\033[92m'+t+'\033[0m'
def er(t):return '\033[91m'+t+'\033[0m'
def dim(t):return '\033[90m'+t+'\033[0m'
def hdr(title):print('\n'+'='*62);print('  '+title);print('='*62)
def auto_lang(fn):
 fn=fn.lower()
 for k,c in {'[id]':'id','indonesian':'id','indo':'id','[en]':'en','english':'en','[ja]':'ja','japanese':'ja','jpn':'ja','[ko]':'ko','[zh]':'zh'}.items():
  if k in fn:return c
 return 'und'

def norm_lang(code,fallback_fn):
 code=str(code or '').strip().lower()
 m3={'jpn':'ja','eng':'en','ind':'id','kor':'ko','chi':'zh','zho':'zh','msa':'ms','ara':'ar','ger':'de','deu':'de','fre':'fr','fra':'fr','spa':'es','por':'pt','rus':'ru','ita':'it','tha':'th','vie':'vi','hin':'hi','und':'und'}
 if code in m3:return m3[code]
 full={'japanese':'ja','english':'en','indonesian':'id','korean':'ko','chinese':'zh','malay':'ms','arabic':'ar','german':'de','french':'fr','spanish':'es','portuguese':'pt','russian':'ru','italian':'it','thai':'th','vietnamese':'vi','hindi':'hi'}
 if code in full:return full[code]
 if code in L:return code
 if len(code)>3:return auto_lang(code)
 return code if code else 'und'

def probe_file(f):
 f=Path(f)
 tracks=[]
 # Primary: mkvmerge -J (JSON, akurat: semua track + bahasa asli file)
 try:
  r=subprocess.run(['mkvmerge','-J',str(f)],capture_output=True,text=True,timeout=30)
  if r.returncode==0 and r.stdout.strip():
   data=json.loads(r.stdout)
   nchap=len(data.get('chapters',[]))
   for tr in data.get('tracks',[]):
    ttype=str(tr.get('type','')).lower()
    if ttype=='subtitles':ttype='subtitle'
    codec=str(tr.get('codec',''))
    props=tr.get('properties',{}) or {}
    lang=norm_lang(props.get('language','und'),f.name)
    if lang=='und':lang=auto_lang(f.name)
    nm=str(props.get('track_name','') or '')
    deft='yes' if props.get('default_track',False) else 'no'
    tracks.append({'file':str(f),'file_name':f.name,'file_type':_det_type(f),'track_id':int(tr.get('id',0)),'codec':codec,'type':ttype,'language':lang,'default':deft,'forced':'yes' if props.get('forced_track',False) else 'no','delay':0,'name':nm,'enabled':True,'chapters':nchap})
   if tracks:return tracks
 except Exception as e:dbg=str(e)[:120]
 # Fallback: --identify (format: Track ID 0: video (AV1) -> grup2=TIPE, grup3=CODEC)
 try:
  r2=subprocess.run(['mkvmerge','--identify',str(f)],capture_output=True,text=True,timeout=30)
  txt=r2.stdout+'\n'+r2.stderr
  for line in txt.splitlines():
   m=re.match(r'\s*Track ID\s+(\d+):\s+(\w+)\s+\(([^)]+)\)',line)
   if m:
    tid=int(m.group(1))
    if not any(x['track_id']==tid for x in tracks):
     ttype=m.group(2).strip().lower()
     if ttype=='subtitles':ttype='subtitle'
     tracks.append({'file':str(f),'file_name':f.name,'file_type':_det_type(f),'track_id':tid,'codec':m.group(3).strip(),'type':ttype,'language':auto_lang(f.name),'default':'yes' if ttype=='video' else 'no','forced':'no','delay':0,'name':'','enabled':True,'chapters':0})
  if tracks:return tracks
  print('  DEBUG mkvmerge tidak kenal format file ini. Output: '+txt[:300])
 except Exception as e2:print('  DEBUG probe gagal: '+str(e2)[:200])
 return tracks

def _det_type(f):
 e=Path(f).suffix.lower()
 if e in V:return 'video'
 if e in A:return 'audio'
 if e in S:return 'subtitle'
 return 'other'

def scan_files(d):
 fs=[]
 if not d.exists():return fs
 for p in sorted(d.rglob('*')):
  if p.is_file():
   e=p.suffix.lower()
   if e in V:fs.append(('video',p))
   elif e in A:fs.append(('audio',p))
   elif e in S:fs.append(('subtitle',p))
 return fs

def _ico(t):return {'video':'V','audio':'A','subtitle':'S'}.get(t,'?')

def load_tracks(sel_files):
 all_tracks=[]
 for ftype,fp in sel_files:
  tracks=probe_file(fp)
  if not tracks:
   all_tracks.append({'file':str(fp),'file_name':fp.name,'file_type':ftype,'track_id':0,'codec':ftype,'type':ftype,'language':auto_lang(fp.name),'default':'yes' if ftype=='video' else 'no','forced':'no','delay':0,'name':'','enabled':True})
  else:
   all_tracks.extend(tracks)
 for i,t in enumerate(all_tracks):t['global_idx']=i
 return all_tracks

def _pad(s,w):
 s=str(s)
 if len(s)>w:return s[:w-2]+'..'
 return s+(' '*(w-len(s)))

def show_tracks(all_tracks):
 print()
 print('  '+_pad('No',2)+'  '+_pad('Codec',20)+'  '+_pad('Type',8)+'  '+_pad('Lang',4)+'  '+_pad('Name',30)+'  '+_pad('TID',3)+'  Def  Copy')
 print('  '+'-'*76)
 by_file={}
 for t in all_tracks:
  by_file.setdefault(t['file'],[]).append(t)
 for filepath,tracks in by_file.items():
  fname=tracks[0]['file_name']
  ch=tracks[0].get('chapters',0)
  chs='  '+str(ch)+' chapters' if ch else ''
  print('  ['+_ico(tracks[0]['file_type'])+'] '+fname+' ('+str(len(tracks))+' tracks'+chs+')')
  for t in tracks:
   de=ok('Yes') if t['default']=='yes' else dim('No ')
   en=ok('ON ') if t['enabled'] else er('OFF')
   idx=_pad(t['global_idx'],2);co=_pad(t['codec'],20);ty=_pad(t['type'],8);la=_pad(t['language'],4)
   nm=_pad(t['name'] if t['name'] else '-',18);tid=_pad(t['track_id'],3)
   print('  '+idx+'  '+co+'  '+ty+'  '+la+'  '+nm+'  '+tid+'  '+de+'  '+en)
  print()

def edit_track(t,all_tracks):
 while True:
  ci()
  print('\n  EDIT TRACK ['+str(t['global_idx'])+']')
  print('  File: '+t['file_name'])
  print('  Type: '+t['type']+'  Codec: '+t['codec']+'\n')
  print('    [1] Language    : '+t['language']+' ('+L.get(t['language'],'?')+')')
  print('    [2] Default     : '+t['default'])
  print('    [3] Forced      : '+t['forced'])
  print('    [4] Delay       : '+str(t['delay'])+'ms')
  print('    [5] Track Name  : '+(t['name'] or '(kosong)'))
  en_str='Yes' if t['enabled'] else 'No'
  print('    [6] Enabled     : '+en_str)
  print('    [7] Jadikan SATU-SATUNYA default tipe ini')
  print('\n    [0] Kembali\n')
  c=input('  Pilih: ').strip()
  if c=='0':return
  elif c=='1':
   print('\n  Codes: '+', '.join(sorted(L.keys())))
   v=input('  Language ['+t['language']+']: ').strip()
   if v:t['language']=v
  elif c=='2':t['default']='no' if t['default']=='yes' else 'yes'
  elif c=='3':t['forced']='no' if t['forced']=='yes' else 'yes'
  elif c=='4':
   try:t['delay']=int(input('  Delay ['+str(t['delay'])+']: ').strip() or t['delay'])
   except:pass
  elif c=='5':t['name']=input('  Name ['+t['name']+']: ').strip()
  elif c=='6':t['enabled']=not t['enabled']
  elif c=='7':
   for o in all_tracks:
    if o['type']==t['type']:o['default']='no'
   t['default']='yes'
   print('  Track ini sekarang satu-satunya default '+t['type']+'.')
   input('  Enter...')

def build_cmd(all_tracks,out):
 cmd=['mkvmerge','-o',str(out)]
 by_file={}
 for t in all_tracks:
  if not t['enabled']:continue
  by_file.setdefault(t['file'],[]).append(t)
 for filepath,tracks in by_file.items():
  cmd.extend(['--no-chapters','--no-global-tags'])
  for t in tracks:
   tid=str(t['track_id'])
   tn=t['name']
   if tn:cmd.extend(['--track-name',tid+':'+tn])
   tl=t['language']
   if tl and tl!='und':cmd.extend(['--language',tid+':'+tl])
   cmd.extend(['--default-track',tid+':'+t['default']])
   if t['forced']=='yes':cmd.extend(['--forced-track',tid+':yes'])
   if t['delay']:cmd.extend(['--sync',tid+':'+str(t['delay'])])
  cmd.append(filepath)
 return cmd

def sel_files():
 ci()
 hdr('PILIH FILE')
 files=scan_files(UPLOAD)
 if not files:
  print('\n  '+er('Tidak ada file di '+str(UPLOAD)))
  print('  Download file dulu lewat menu Download.\n')
  return None
 vids=[(i,f) for i,(t,f) in enumerate(files) if t=='video']
 auds=[(i,f) for i,(t,f) in enumerate(files) if t=='audio']
 subs=[(i,f) for i,(t,f) in enumerate(files) if t=='subtitle']
 print()
 if vids:
  print('  VIDEO:')
  for i,f in vids:
   size=f.stat().st_size/1024/1024
   print('    ['+str(i)+'] '+f.name+'  '+dim(str(int(size))+'MB'))
  print()
 if auds:
  print('  AUDIO:')
  for i,f in auds:
   size=f.stat().st_size/1024/1024
   print('    ['+str(i)+'] '+f.name+'  '+dim(str(int(size))+'MB'))
  print()
 if subs:
  print('  SUBTITLE:')
  for i,f in subs:
   print('    ['+str(i)+'] '+f.name)
  print()
 print('  '+'-'*50)
 print('  Pilih: 0,1,3  atau  0-3  atau  * (semua)')
 print('  '+'-'*50)
 print()
 print('  [Q] Kembali')
 print()
 while True:
  c=input('  > ').strip()
  if not c:continue
  if c.upper()=='Q':return None
  if c=='*':return [(files[i][0],files[i][1]) for i in range(len(files))]
  try:
   nums=[]
   for part in c.split(','):
    part=part.strip()
    if '-' in part:
     a,b=part.split('-',1);nums.extend(range(int(a),int(b)+1))
    else:nums.append(int(part))
   sel=[n for n in nums if 0<=n<len(files)]
   if sel:return [(files[i][0],files[i][1]) for i in sel]
  except:print('  Input tidak valid!')

def load_secrets():
 try:
  if os.path.exists('/content/.haru_secrets.json'):
   d=json.load(open('/content/.haru_secrets.json'))
   for k,v in d.items():
    if v and not os.environ.get(k):os.environ[k]=str(v)
 except:pass
def get_secret(k):
 v=os.environ.get(k,'')
 if v:return v.strip()
 try:
  from google.colab import userdata
  t=userdata.get(k)
  if t:return str(t).strip()
 except:pass
 return ''
def get_gofile_token():
 return get_secret('GOFILE_API_TOKEN')

def gofile_api_generate(url,password,token):
 payload={'url':url,'password':password,'expiresInSeconds':3600,'filePage':0,'fileSize':100}
 headers={'Authorization':'Bearer '+token,'Content-Type':'application/json'}
 r=requests.post('https://go.filmbeehub.workers.dev/api/v1/generate',json=payload,headers=headers,timeout=60)
 return r.json()

def gofile_api_list(url,password,token):
 res=gofile_api_generate(url,password,token)
 if not res.get('ok'):
  print('  Gagal generate: '+str(res.get('error','unknown')))
  return []
 data=res.get('data',{})
 if data.get('downloadLinks'):return data['downloadLinks']
 share_url=data.get('shareUrl','')
 if share_url:
  sid=share_url.rstrip('/').split('/')[-1]
  print('  Share ID: '+sid)
  rr=requests.get('https://go.filmbeehub.workers.dev/api/data/'+sid,headers={'User-Agent':'Mozilla/5.0'},timeout=30)
  fd=rr.json()
  out=[]
  for g in fd.get('groups',[]):out.extend(g.get('files',[]))
  return out
 return []

def gofile_dl_one(link,tries=3):
 durl=link.get('downloadUrl','')
 name=link.get('name','file')
 if not durl:print('  Tidak ada download URL, skip.');return None
 dest=UPLOAD/name
 part=UPLOAD/(name+'.part')
 if dest.exists() and dest.stat().st_size>0:
  print('  SKIP '+name+' (sudah ada)')
  return dest
 for att in range(1,tries+1):
  try:
   print('  Downloading '+name+'...'+('' if att==1 else ' (coba '+str(att)+')'))
   rr=requests.get(durl,stream=True,timeout=600)
   rr.raise_for_status()
   total=0
   fh=open(part,'wb')
   for ch in rr.iter_content(chunk_size=1024*1024):
    if ch:fh.write(ch);total+=len(ch)
   fh.close()
   if total==0:raise Exception('0 byte')
   os.rename(part,dest)
   print('  OK '+name+' ('+str(total)+' bytes / '+str(round(total/1024/1024,1))+' MB)')
   return dest
  except Exception as e:
   try:fh.close()
   except:pass
   try:
    if part.exists():os.remove(part)
   except:pass
   if att<tries:
    wait=10*att
    print('  Gagal, retry '+str(wait)+' detik... ('+str(e)[:120]+')')
    time.sleep(wait)
   else:print(er('  Gagal: '+name+' - '+str(e)[:150]))
 return None

def gofile_wt(agent,token):
 import hashlib,time
 slot=int(time.time())//14400
 return hashlib.sha256((agent+'::en-US::'+token+'::'+str(slot)+'::12af056dacea0b').encode()).hexdigest()

def gofile_direct_fetch(url,password):
 import hashlib
 m=re.search(r'gofile\.io/d/(\w+)',url)
 if not m:return None,'Link tidak valid',None
 cid=m.group(1)
 pw=hashlib.sha256(password.encode()).hexdigest() if password else None
 agent='Mozilla/5.0'
 s=requests.Session()
 s.headers.update({'Accept-Encoding':'gzip','User-Agent':agent,'Connection':'keep-alive','Accept':'*/*','Origin':'https://gofile.io','Referer':'https://gofile.io/'})
 try:
  r=s.post('https://api.gofile.io/accounts',headers={'X-Website-Token':gofile_wt(agent,''),'X-BL':'en-US'},timeout=20)
  tok=r.json()['data']['token']
 except Exception as e:return None,'Guest account gagal: '+str(e)[:120],None
 s.cookies.set('Cookie','accountToken='+tok)
 s.headers.update({'Authorization':'Bearer '+tok})
 files=[]
 try:
  def walk(x):
   u='https://api.gofile.io/contents/'+x+'?cache=true'
   if pw:u=u+'&password='+pw
   r=s.get(u,headers={'X-Website-Token':gofile_wt(agent,tok),'X-BL':'en-US'},timeout=30)
   d=r.json()
   if d.get('status')!='ok':raise Exception(str(d.get('status'))[:60])
   data=d['data']
   if data.get('passwordStatus','passwordOk')!='passwordOk' and 'password' in data:raise Exception('password salah')
   if data.get('type')!='folder':
    if data.get('link'):files.append({'name':data['name'],'size':data.get('size',0),'link':data['link']})
    return
   for ch in (data.get('children',{}) or {}).values():
    if ch.get('type')=='folder':walk(ch['id'])
    elif ch.get('link'):files.append({'name':ch['name'],'size':ch.get('size',0),'link':ch['link']})
  walk(cid)
 except Exception as e:return None,'List gagal: '+str(e)[:150],None
 return files,None,tok

def gofile_direct_one(f,tok,dest_dir):
 name=f['name'];dest=dest_dir/name;part=dest_dir/(name+'.part')
 if dest.exists() and dest.stat().st_size>0:
  print('  SKIP '+name+' (sudah ada)');return True
 hdr={'User-Agent':'Mozilla/5.0','Referer':'https://gofile.io/','Origin':'https://gofile.io','Cookie':'accountToken='+tok}
 for att in range(1,4):
  try:
   print('  Direct '+name+'...'+('' if att==1 else ' (coba '+str(att)+')'))
   rr=requests.get(f['link'],headers=hdr,stream=True,timeout=600)
   rr.raise_for_status()
   total=0
   fh=open(part,'wb')
   for ch in rr.iter_content(chunk_size=1024*1024):
    if ch:fh.write(ch);total+=len(ch)
   fh.close()
   if total==0:raise Exception('0 byte')
   os.rename(part,dest)
   print('  OK '+name+' ('+str(total)+' bytes / '+str(round(total/1024/1024,1))+' MB)')
   return True
  except Exception as e:
   try:fh.close()
   except:pass
   try:
    if part.exists():os.remove(part)
   except:pass
   if att<3:
    print('  Gagal, retry... ('+str(e)[:120]+')')
    time.sleep(10*att)
   else:print(er('  Gagal: '+name+' - '+str(e)[:150]))
 return False

def gofile_direct_retry(url,pwd,names,dest_dir):
 print('  Coba jalur direct API untuk '+str(len(names))+' file...')
 files,err,tok=gofile_direct_fetch(url,pwd)
 if err:print(er('  Direct: '+err));return names
 targets=[f for f in files if f['name'] in names]
 if not targets:print(er('  Direct: file tidak ketemu di listing.'));return names
 still=[]
 for f in targets:
  if not gofile_direct_one(f,tok,dest_dir):still.append(f['name'])
 return still



def dl_gofile():
 print('  Redirecting ke haru-download...');subprocess.run(['haru-download'])

def dl_gofile():
 print('  Redirecting ke haru-download...');subprocess.run(['haru-download'])
def dl_drive():dl_gofile()

def dl_gofile():
 print('  Redirecting ke haru-download...');subprocess.run(['haru-download'])
def dl_drive():dl_gofile()
def dl_url():dl_gofile()

def dl_gofile():
 print('  Redirecting ke haru-download...');subprocess.run(['haru-download'])
def dl_drive():dl_gofile()
def dl_url():dl_gofile()
def menu_download():dl_gofile()

def menu_download():dl_gofile()

def dl_url():dl_gofile()
def menu_download():dl_gofile()

def dl_drive():
 hdr('DOWNLOAD - Google Drive')
 url=input('\n  Link GDrive: ').strip()
 if not url:return
 print('  Downloading...')
 r=subprocess.run(['gdown','--folder','-O',str(UPLOAD),'--remaining-ok',url],timeout=300)
 if r.returncode==0:print(ok('  Download selesai!'))
 else:print(er('  Download gagal (code '+str(r.returncode)+')'))


def dl_url():
 hdr('DOWNLOAD - Direct URL')
 url=input('\n  Direct URL: ').strip()
 if not url:return
 fname=input('  Filename (kosong = auto): ').strip() or None
 cmd=['wget','-q','-P',str(UPLOAD),'--content-disposition','--no-check-certificate']
 if fname:cmd.extend(['-O',str(UPLOAD/fname)])
 cmd.append(url)
 r=subprocess.run(cmd,timeout=300)
 if r.returncode==0:print(ok('  Download selesai!'))
 else:print(er('  Download gagal (code '+str(r.returncode)+')'))


def menu_download():
 ci();hdr('DOWNLOAD')
 print()
 print('  [1] Gofile')
 print('  [2] Google Drive')
 print('  [3] Direct URL')
 print()
 print('  [0] Kembali')
 print()
 c=input('  Pilih: ').strip()
 if c=='0':return
 elif c=='1':dl_gofile()
 elif c=='2':dl_drive()
 elif c=='3':dl_url()

def get_default_output(all_tracks):
 # Cari video file pertama, pakai namafilenya
 for t in all_tracks:
  if t['file_type']=='video':
   name=Path(t['file']).stem
   return OUTPUT/(name+'.mkv')
 return OUTPUT/'output.mkv'

def fix_defaults(all_tracks):
 notes=[]
 for tt in ['video','audio','subtitle']:
  ds=[t for t in all_tracks if t['enabled'] and t['type']==tt and t['default']=='yes']
  if len(ds)>1:
   for t in ds[1:]:t['default']='no'
   notes.append(tt+': keep #'+str(ds[0]['global_idx'])+' ('+ds[0]['language']+'), reset '+str(len(ds)-1)+' lain -> No')
 return notes

def summ_output(out):
 try:
  r=subprocess.run(['mkvmerge','-J',str(out)],capture_output=True,text=True,timeout=30)
  data=json.loads(r.stdout)
  by={}
  for tr in data.get('tracks',[]):
   tt=str(tr.get('type',''));pr=tr.get('properties',{}) or {}
   by.setdefault(tt,[]).append(str(pr.get('language','und'))+(' [DEF]' if pr.get('default_track',False) else ''))
  for tt,ls in by.items():print('    '+tt+': '+str(len(ls))+' track ('+', '.join(ls)+')')
 except:pass

def menu_mux():
 while True:
  sel=sel_files()
  if not sel:input('  Enter...');return
  all_tracks=load_tracks(sel)
  if not all_tracks:print(er('  Tidak ada track.'));input('  Enter...');return
  out=get_default_output(all_tracks)
  while True:
   ci();hdr('TRACK EDITOR')
   ec=sum(1 for t in all_tracks if t['enabled'])
   show_tracks(all_tracks)
   print('  [0-9]  Edit track (pilih angka)')
   print('  [D#]   Toggle default (contoh: D2)')
   print('  [E#]   Toggle enable/disable (contoh: E3)')
   print('  [S]    Output filename')
   print('  [M]    Mux!')
   print('  [Q]    Kembali')
   print('\n  Output: '+out.name+'  |  Active: '+str(ec)+'/'+str(len(all_tracks))+' tracks')
   print()
   c=input('  > ').strip().upper()
   if c=='Q':break
   elif c=='S':
    v=input('  Filename ['+out.name+']: ').strip()
    if v:out=out.parent/v
   elif c=='M':
    en=[t for t in all_tracks if t['enabled']]
    if not en:print(er('  No active tracks!'));input('  Enter...');continue
    notes=fix_defaults(all_tracks)
    if notes:
     print('  Auto-fix default (1 per tipe):')
     for nn in notes:print('    '+nn)
    cmd=build_cmd(all_tracks,out)
    print('\n  Muxing '+str(len(en))+' tracks -> '+out.name+' ...\n')
    r=subprocess.run(cmd,capture_output=True,text=True,timeout=600)
    if out.exists() and out.stat().st_size>0:
     mb=out.stat().st_size/1024/1024
     print(ok('  SELESAI: '+out.name+' ('+str(round(mb,1))+' MB)'))
     tg_send('<b>Mux selesai</b>\n'+out.name+' ('+str(round(mb,1))+' MB)')
     print('  Isi file hasil:')
     summ_output(out)
     ws=[l for l in r.stdout.splitlines() if 'Warning' in l]
     if ws:
      print('  '+str(len(ws))+' warnings:')
      for w in ws[:5]:print('    '+w[:120])
    else:print(er('  Failed! '+r.stderr[-500:]))
    input('\n  Enter...');break
   elif c.startswith('D') and len(c)>1:
    try:
     i=int(c[1:])
     idx=[t['global_idx'] for t in all_tracks].index(i)
     t=all_tracks[idx]
     t['default']='no' if t['default']=='yes' else 'yes'
    except:pass
   elif c.startswith('E') and len(c)>1:
    try:
     i=int(c[1:])
     idx=[t['global_idx'] for t in all_tracks].index(i)
     all_tracks[idx]['enabled']=not all_tracks[idx]['enabled']
    except:pass
   elif c.isdigit():
    i=int(c)
    try:
     idx=[t['global_idx'] for t in all_tracks].index(i)
     edit_track(all_tracks[idx],all_tracks)
    except:pass

def ep_key(name):
 import re
 s=name.lower()
 for p in [r's\d{1,2}e(\d{1,3})',r'\be(?:p|isode)?[\s._-]*(\d{1,3})',r'\[(\d{1,3})\]',r'[\s._-](\d{1,3})[\s._-]']:
  m=re.search(p,s)
  if m:
   v=m.group(1).lstrip('0')
   return v if v else '0'
 return ''

def menu_batch():
 while True:
  ci();hdr('BATCH SERIES MUX')
  vids=[];subs=[];auds=[]
  for d in [UPLOAD,OUTPUT,Path('/content/extracts')]:
   if not d.exists():continue
   for p in sorted(d.rglob('*')):
    if not p.is_file():continue
    e=p.suffix.lower()
    if e in V:vids.append(p)
    elif e in S:subs.append(p)
    elif e in A:auds.append(p)
  if not vids or (not subs and not auds):
   print(er('  Butuh video + (subtitle/audio) di folder.'));input('  Enter...');return
  byv={};bys={};bya={}
  for p in vids:byv.setdefault(ep_key(p.name),[]).append(p)
  for p in subs:bys.setdefault(ep_key(p.name),[]).append(p)
  for p in auds:bya.setdefault(ep_key(p.name),[]).append(p)
  ekeys=sorted(set(byv)&(set(bys)|set(bya)),key=lambda x:int(x) if x.isdigit() else 9999)
  pairs=[]
  for k in ekeys:
   if k=='':continue
   pairs.append((k,byv[k][0],bys.get(k,[]),bya.get(k,[])))
  lone_v=[(k,byv[k][0].name) for k in sorted(set(byv)-(set(bys)|set(bya))) if k!='']
  lone_s=[(k,bys[k][0].name) for k in sorted(set(bys)-set(byv)) if k!='']
  lone_a=[(k,bya[k][0].name) for k in sorted(set(bya)-set(byv)) if k!='']
  if not pairs:
   print(er('  Tidak ada pasangan episode cocok.'));input('  Enter...');return
  print()
  print('  No  EP   Video                   Sub/Aud files')
  print('  '+'-'*72)
  for i,(k,v,ss,aa) in enumerate(pairs):
   extra='+'.join([s.name[:20] for s in (ss+aa)][:3])
   if len(ss)+len(aa)>3:extra=extra+'+...'
   print('  '+str(i)+'   '+k+'   '+v.name[:26]+'  '+extra)
  print()
  if lone_v or lone_s or lone_a:
   print('  Tanpa pasangan (di-skip):')
   for k,n in lone_v:print('    EP '+k+' video: '+n[:50])
   for k,n in lone_s:print('    EP '+k+' sub: '+n[:50])
   for k,n in lone_a:print('    EP '+k+' audio: '+n[:50])
   print()
  dlang_s=input('  Bahasa default untuk SUB yg und [id]: ').strip() or 'id'
  dlang_a=input('  Bahasa default untuk AUDIO yg und (kosong=biarkan): ').strip()
  print()
  print('  [Y] Gas mux semua   [nomor] buang pair (0,2)   [B] Bulk edit tracks   [Q] batal')
  print()
  c=input('  > ').strip().upper()
  if c=='Q':return
  if c=='B':
   batch_track_edit(pairs,dlang_s,dlang_a)
   continue
  if c!='Y':
   try:
    drop=set()
    for part in c.split(','):
     part=part.strip()
     if part.isdigit():drop.add(int(part))
    pairs=[p for i,p in enumerate(pairs) if i not in drop]
   except:return
   if not pairs:return
  ok_n=0;fail=[];done_names=[]
  for k,v,ss,aa in pairs:
   sel=[('video',v)]+[('subtitle',s) for s in ss]+[('audio',s) for s in aa]
   all_tracks=load_tracks(sel)
   for t in all_tracks:
    if t['type']=='subtitle':
     if t['language']=='und':t['language']=dlang_s
     t['default']='no'
    if t['type']=='audio' and t['language']=='und' and dlang_a:t['language']=dlang_a
   for t in all_tracks:
    if t['type']=='subtitle' and Path(t['file']).suffix.lower() in S:
     t['default']='yes'
     break
   notes=fix_defaults(all_tracks)
   out=OUTPUT/(v.stem+'.mkv')
   cmd=build_cmd(all_tracks,out)
   print('\n  ['+k+'] Muxing -> '+out.name+' ...')
   r=subprocess.run(cmd,capture_output=True,text=True,timeout=600)
   if out.exists() and out.stat().st_size>0:
    mb=out.stat().st_size/1024/1024
    print('  '+ok('OK')+' '+out.name+' ('+str(round(mb,1))+' MB)')
    ok_n+=1
    done_names.append(out.name)
   else:
    print('  '+er('GAGAL')+' '+v.name)
    fail.append(v.name)
  print('\n  Selesai: '+str(ok_n)+'/'+str(len(pairs))+' episode.')
  if fail:print('  Gagal: '+', '.join(fail)[:200])
  msg='<b>Batch mux selesai</b>\n'+str(ok_n)+'/'+str(len(pairs))+' episode'
  if done_names:msg=msg+'\n'+'\n'.join(done_names[:15])
  if len(done_names)>15:msg=msg+'\n... +'+str(len(done_names)-15)+' lagi'
  tg_send(msg)
  input('\n  Enter...')

def batch_track_edit(pairs,dlang_s,dlang_a):
 all_files=[]
 for k,v,ss,aa in pairs:
  all_files.append(('video',v,k))
  for s in ss:all_files.append(('subtitle',s,k))
  for a in aa:all_files.append(('audio',a,k))
 if not all_files:return
 ci();hdr('BATCH TRACK EDITOR')
 print('  Combined tracks dari '+str(len(pairs))+' episode:')
 print()
 print('  No  Type      Lang  Name                    EP   File')
 print('  '+'-'*72)
 for i,(typ,f,ep) in enumerate(all_files):
  name=str(getattr(f,'stem',''))[:20]
  lang=auto_lang(f.name)
  print('  '+str(i).ljust(3)+typ.ljust(10)+lang.ljust(6)+name[:20].ljust(22)+ep.ljust(5)+f.name[:30])
 print()
 print('  Filter: [A]udio  [S]ubtitle  [V]ideo  [ALL] Semua')
 print('  [D#] Set delay (contoh: D0=1500 -> set delay 1500ms ke track 0)')
 print('  [N#] Set name  (contoh: N0=Indonesian -> rename track 0)')
 print('  [DA] Delay ALL filtered tracks')
 print('  [NA] Name ALL filtered tracks')
 print('  [Q]  Kembali')
 print()
 filt=None
 filtered=list(range(len(all_files)))
 while True:
  c=input('  > ').strip().upper()
  if c=='Q':return
  if c=='A':filt='audio';filtered=[i for i,(t,_,_) in enumerate(all_files) if t=='audio'];print('  Filter: audio ('+str(len(filtered))+')');continue
  if c=='S':filt='subtitle';filtered=[i for i,(t,_,_) in enumerate(all_files) if t=='subtitle'];print('  Filter: subtitle ('+str(len(filtered))+')');continue
  if c=='V':filt='video';filtered=[i for i,(t,_,_) in enumerate(all_files) if t=='video'];print('  Filter: video ('+str(len(filtered))+')');continue
  if c=='ALL':filt=None;filtered=list(range(len(all_files)));print('  Filter: all ('+str(len(filtered))+')');continue
  if c.startswith('DA'):
   v=c[2:].strip()
   if not v:v=input('  Delay value (ms): ').strip()
   if v:
    for i in filtered:
     f=all_files[i][1]
     if not hasattr(f,'delay'):f.delay=0
     try:f.delay=int(v)
     except:pass
    print('  Delay set ke '+v+'ms untuk '+str(len(filtered))+' tracks')
   continue
  if c.startswith('NA'):
   v=c[2:].strip()
   if not v:v=input('  Name value: ').strip()
   if v:
    for i in filtered:
     f=all_files[i][1]
     f.custom_name=v
    print('  Name set ke "'+v+'" untuk '+str(len(filtered))+' tracks')
   continue
  if c.startswith('D') and len(c)>1:
   parts=c[1:].split('=',1)
   if len(parts)==2:
    try:
     idx=int(parts[0]);val=int(parts[1])
     if 0<=idx<len(all_files):
      f=all_files[idx][1]
      if not hasattr(f,'delay'):f.delay=0
      f.delay=val
      print('  Track '+str(idx)+' delay -> '+str(val)+'ms')
    except:pass
   continue
  if c.startswith('N') and len(c)>1:
   parts=c[1:].split('=',1)
   if len(parts)==2:
    try:
     idx=int(parts[0]);val=parts[1]
     if 0<=idx<len(all_files):
      f=all_files[idx][1]
      f.custom_name=val
      print('  Track '+str(idx)+' name -> "'+val+'"')
    except:pass
   continue



def menu_list():
 sel=sel_files()
 if not sel:input('  Enter...');return
 all_tracks=load_tracks(sel)
 if not all_tracks:print(er('  Tidak ada track.'));input('  Enter...');return
 ci();hdr('LIST TRACKS')
 show_tracks(all_tracks)
 input('  Enter...')

def page_out(text):
 ls=text.splitlines()
 if len(ls)>80:
  import tempfile
  tp=os.path.join(tempfile.gettempdir(),'mi.txt')
  open(tp,'w',encoding='utf-8',errors='ignore').write(text)
  print('  Output panjang ('+str(len(ls))+' baris) -> less, q keluar, panah scroll')
  try:subprocess.run(['less','-M','-X',tp])
  except:print(text)
 else:print(text)

def telegraph_upload(title,text):
 try:
  r=requests.post('https://api.telegra.ph/createAccount',data={'short_name':'haru','author_name':'haru-mux'},timeout=20)
  tok=r.json()['result']['access_token']
 except Exception as e:print(er('  Telegraph account gagal: '+str(e)[:120]));return None
 try:
  nodes=json.dumps([{'tag':'pre','children':[text[:60000]]}])
  r=requests.post('https://api.telegra.ph/createPage',data={'access_token':tok,'title':title[:60],'author_name':'haru-mux','content':nodes},timeout=30)
  d=r.json()
  if d.get('ok'):
   url=d['result']['url']
   print(ok('  '+url))
   return url
  print(er('  Telegraph gagal: '+str(d)[:150]))
 except Exception as e:print(er('  Telegraph error: '+str(e)[:120]))
 return None

def telegraph_bulk(title,sections,author):
 pages=[];cur=[];curlen=0
 for name,text in sections:
  bl=len(name)+len(text)+100
  if cur and curlen+bl>58000:
   pages.append(cur);cur=[];curlen=0
  cur.append((name,text));curlen+=bl
 if cur:pages.append(cur)
 urls=[]
 for i,pg in enumerate(pages):
  nodes=[]
  for name,text in pg:
   nodes.append({'tag':'h4','children':[name]})
   nodes.append({'tag':'pre','children':[text[:60000]]})
  try:
   r=requests.post('https://api.telegra.ph/createAccount',data={'short_name':'haru','author_name':author},timeout=20)
   tok=r.json()['result']['access_token']
   t=title+(' (%d/%d)'%(i+1,len(pages)) if len(pages)>1 else '')
   r=requests.post('https://api.telegra.ph/createPage',data={'access_token':tok,'title':t[:60],'author_name':haru-mux,'content':json.dumps(nodes)},timeout=30)
   d=r.json()
   if d.get('ok'):urls.append(d['result']['url']);print(ok('  Hal '+str(i+1)+': '+d['result']['url']))
  except Exception as e:print(er('  Gagal hal '+str(i+1)))
 return urls

def menu_info():
 ci();hdr('MEDIAINFO')
 dirs=[UPLOAD,OUTPUT,Path('/content/extracts')]
 items=[]
 for d in dirs:
  if d.exists():
   for f in sorted(d.rglob('*')):
    if f.is_file() and f.suffix.lower() in V|A|S:items.append((d,f))
 if not items:
  print(er('  Tidak ada file.'));input('  Enter...');return
 print()
 idx=0
 for d in dirs:
  grp=[f for dd,f in items if dd==d]
  if not grp:continue
  print('  ['+d.name+'/]  ('+str(len(grp))+' file)')
  for f in grp:
   size=f.stat().st_size/1024/1024
   print('  ['+str(idx)+'] '+f.name+'  '+dim(str(int(size))+'MB'))
   idx+=1
  print()
 flat=[f for dd,f in items]
 c=input('  Pilih file (* semua / F bulk folder): ').strip()
 if c.upper()=='F':return mi_bulk()
 if c=='*':targets=flat
 else:
  try:
   idx=int(c)
   if 0<=idx<len(flat):targets=[flat[idx]]
   else:return
  except:return
 fmt=input('  Format (T=text, J=json) [T]: ').strip().upper() or 'T'
 ci();hdr('MEDIAINFO - '+targets[0].name)
 saved=[]
 for f in targets:
  cmd=['mediainfo']
  if fmt=='J':cmd.append('--Output=JSON')
  cmd.append(str(f))
  r=subprocess.run(cmd,capture_output=True,text=True,timeout=30)
  page_out(r.stdout)
  saved.append((f.name,r.stdout))
 if saved:
  u=input('\n  Upload ke telegra.ph? [Y/n]: ').strip().lower()
  if u in ('','y'):
   links=[]
   for name,text in saved:
    print('  Upload '+name+'...')
    url=telegraph_upload('MediaInfo - '+name,text)
    if url:links.append((name,url))
   if links:
    msg='<b>MediaInfo</b>'
    for name,url in links:msg=msg+'\n'+name+'\n'+url
    tg_send(msg)
 input('  Enter...')

def upload_gofile():
 print('  Redirecting ke haru-upload...');subprocess.run(['haru-upload'])

def upload_gofile():
 print('  Redirecting ke haru-upload...');subprocess.run(['haru-upload'])
def upload_drive():upload_gofile()

def upload_gofile():
 print('  Redirecting ke haru-upload...');subprocess.run(['haru-upload'])
def upload_drive():upload_gofile()
def menu_upload():upload_gofile()

def menu_upload():upload_gofile()

def gdrive_secret(k):
 return get_secret(k)

def gdrive_token(cid,sec,ref):
 try:
  r=requests.post('https://oauth2.googleapis.com/token',data={'client_id':cid,'client_secret':sec,'refresh_token':ref,'grant_type':'refresh_token'},timeout=15)
  return r.json().get('access_token')
 except:return None

def parse_drive_folder(tok,folder):
 import re
 m=re.search(r'/folders/([A-Za-z0-9_-]+)',folder)
 if m:return m.group(1)
 if len(folder)>20 and '/' not in folder and ' ' not in folder:return folder
 if tok:return gdrive_find_folder(tok,folder)
 return None

def gdrive_find_folder(tok,name):
 try:
  q="name='"+name+"' and mimeType='application/vnd.google-apps.folder' and trashed=false"
  r=requests.get('https://www.googleapis.com/drive/v3/files',headers={'Authorization':'Bearer '+tok},params={'q':q,'fields':'files(id,name)'},timeout=15)
  fs=r.json().get('files',[])
  if fs:return fs[0]['id']
  meta={'name':name,'mimeType':'application/vnd.google-apps.folder'}
  r2=requests.post('https://www.googleapis.com/drive/v3/files',headers={'Authorization':'Bearer '+tok,'Content-Type':'application/json'},data=json.dumps(meta),timeout=15)
  return r2.json().get('id')
 except:return None

def gdrive_upload_file(tok,fpath,parent):
 size=fpath.stat().st_size
 meta={'name':fpath.name,'parents':[parent]}
 try:
  r=requests.post('https://www.googleapis.com/upload/drive/v3/files?uploadType=resumable',headers={'Authorization':'Bearer '+tok,'Content-Type':'application/json','X-Upload-Content-Type':'application/octet-stream','X-Upload-Content-Length':str(size)},data=json.dumps(meta),timeout=30)
  uri=r.headers.get('Location')
  if not uri:print('  Gagal mulai sesi upload.');return False
 except Exception as e:print('  Error inisiasi: '+str(e)[:150]);return False
 CH=64*1024*1024 if size>100*1024*1024 else 16*1024*1024
 up=0;t0=time.time()
 try:
  fh=open(fpath,'rb')
  while up<size:
   ch=fh.read(CH)
   if not ch:break
   end=up+len(ch)-1
   rr=requests.put(uri,headers={'Content-Range':'bytes '+str(up)+'-'+str(end)+'/'+str(size),'Content-Length':str(len(ch))},data=ch,timeout=120)
   if rr.status_code in (200,201):up+=len(ch);break
   elif rr.status_code==308:
    up+=len(ch)
    el=time.time()-t0;sp=up/el/1024/1024 if el>0 else 0
    print('  '+str(round(up/size*100,1))+'%  '+str(round(sp,1))+' MB/s')
   else:print('  Upload error HTTP '+str(rr.status_code));fh.close();return False
  fh.close()
 except Exception as e:print('  Error upload: '+str(e)[:150]);return False
 print(ok('  100% Selesai.'))
 return True


def upload_drive():
 hdr('UPLOAD - Google Drive')
 all_files=[]
 for d in [UPLOAD,OUTPUT,Path('/content/extracts'),Path('/content/downloads')]:
  if d.exists():
   for f in sorted(d.rglob('*')):
    if f.is_file() and f.suffix.lower() in V|A|S:all_files.append((d,f))
 if not all_files:print(er('  Tidak ada file untuk di-upload.'));input('  Enter...');return
 print()
 idx=0
 for d in [UPLOAD,OUTPUT,Path('/content/extracts'),Path('/content/downloads')]:
  grp=[(dd,f) for dd,f in all_files if dd==d]
  if not grp:continue
  print('  ['+d.name+'/]  ('+str(len(grp))+' file)')
  for dd,f in grp:
   size=f.stat().st_size/1024/1024
   print('    ['+str(idx)+'] '+f.name+'  '+dim(str(int(size))+'MB'))
   idx+=1
  print()
 flat=[f for dd,f in all_files]
 c=input('  Pilih (* semua / 0,1,2 / 0-3 / Q batal): ').strip().upper()
 if c=='Q':return
 if c=='*':targets=flat
 else:
  try:
   nums=[]
   for part in c.split(','):
    part=part.strip()
    if '-' in part:a,b=part.split('-',1);nums.extend(range(int(a),int(b)+1))
    else:nums.append(int(part))
   targets=[flat[n] for n in nums if 0<=n<len(flat)]
  except:print('  Input tidak valid.');input('  Enter...');return
  if not targets:return
 cid=gdrive_secret('GDRIVE_CLIENT_ID');sec=gdrive_secret('GDRIVE_CLIENT_SECRET');ref=gdrive_secret('GDRIVE_REFRESH_TOKEN')
 parent_id=gdrive_secret('GDRIVE_FOLDER_ID') or '1pjpd63PTFvwYd8iI7dvMwcU-e_LMqvUE'
 if not(cid and sec and ref):
  print(er('  Secret GDrive tidak kebaca.'));print('  Aktifkan toggle secret + re-run cell Install.');input('  Enter...');return
 print('  Auth via API...')
 tok=gdrive_token(cid,sec,ref)
 if not tok:print(er('  Gagal dapat access token.'));return
 import re
 m=re.search(r'/folders/([A-Za-z0-9_-]+)',parent_id)
 if m:parent_id=m.group(1)
 elif len(parent_id)<20:
  q="name='"+parent_id+"' and mimeType='application/vnd.google-apps.folder' and trashed=false"
  try:
   r=requests.get('https://www.googleapis.com/drive/v3/files',headers={'Authorization':'Bearer '+tok},params={'q':q,'fields':'files(id)'},timeout=15)
   fs=r.json().get('files',[])
   if fs:parent_id=fs[0]['id']
  except:pass
 sub=input('  Subfolder ['+dim('langsung ke parent')+']: ').strip()
 target=parent_id
 if sub:
  try:
   q2="name='"+sub+"' and '"+parent_id+"' in parents and mimeType='application/vnd.google-apps.folder' and trashed=false"
   r2=requests.get('https://www.googleapis.com/drive/v3/files',headers={'Authorization':'Bearer '+tok},params={'q':q2,'fields':'files(id)'},timeout=15)
   fs2=r2.json().get('files',[])
   if fs2:target=fs2[0]['id']
   else:
    meta={'name':sub,'mimeType':'application/vnd.google-apps.folder','parents':[parent_id]}
    r3=requests.post('https://www.googleapis.com/drive/v3/files',headers={'Authorization':'Bearer '+tok,'Content-Type':'application/json'},data=json.dumps(meta),timeout=15)
    nid=r3.json().get('id')
    if nid:target=nid;print('  Subfolder dibuat: '+sub)
    else:print(er('  Gagal buat subfolder.'))
  except:print(er('  Error buat subfolder.'))
 ok_n=0;fail=[]
 for f in targets:
  print('  Upload '+f.name+' ('+str(round(f.stat().st_size/1024/1024,1))+'MB)...')
  if gdrive_upload_file(tok,f,target):ok_n+=1;print('  '+ok('ok')+' '+f.name)
  else:fail.append(f.name);print('  '+er('gagal')+' '+f.name)
 if ok_n:tg_send('<b>Upload GDrive</b>\n'+str(ok_n)+' file berhasil')
 if fail:print(er('  Gagal: '+', '.join(fail)))
 input('\n  Enter...')


def mi_bulk():
 ci();hdr('BULK MEDIAINFO')
 dirs=[d for d in [UPLOAD,OUTPUT,Path('/content/extracts')] if d.exists()]
 if not dirs:return
 print()
 for i,d in enumerate(dirs):print('  ['+str(i)+'] '+str(d))
 print()
 c=input('  Folder: ').strip()
 try:d=dirs[int(c)]
 except:return
 fs=[p for p in sorted(d.rglob('*')) if p.is_file() and p.suffix.lower() in V|A|S]
 if not fs:print(er('  Kosong.'));input('  Enter...');return
 print('\n  Proses '+str(len(fs))+' file...')
 sections=[]
 for f in fs:
  r=subprocess.run(['mediainfo',str(f)],capture_output=True,text=True,timeout=30)
  sections.append((f.name,r.stdout))
  print('  ok '+f.name)
 print()
 urls=telegraph_bulk('MediaInfo - '+d.name+' ('+str(len(fs))+' file)',sections,'haru-mux')
 if urls:
  msg='<b>Bulk MediaInfo</b>\n'+str(len(fs))+' file'
  for u in urls:msg=msg+'\n'+u
  tg_send(msg)
 input('\n  Enter...')


def menu_upload():
 ci();hdr('UPLOAD')
 print()
 print('  [1] Gofile  (folder gabungan)')
 print('  [2] Google Drive (multi-file + subfolder)')
 print()
 print('  [0] Kembali')
 print()
 c=input('  Pilih: ').strip()
 if c=='0':return
 elif c=='1':upload_gofile()
 elif c=='2':upload_drive()


def menu_delete():
 ci();hdr('HAPUS FILE')
 import shutil
 roots=[UPLOAD,OUTPUT,Path('/content/extracts'),Path('/content/downloads')]
 files=[]
 for d in roots:
  if d.exists():
   for p in sorted(d.rglob('*')):
    if p.is_file():files.append((d,p))
 if not files:print(er('  Semua folder kosong.'));input('  Enter...');return
 idx=0
 for d in roots:
  grp=[p for dd,p in files if dd==d]
  if not grp:continue
  print('  ['+d.name+'/]  ('+str(len(grp))+' file)')
  for p in grp:
   size=p.stat().st_size/1024/1024
   print('  ['+str(idx)+'] '+p.name+'  '+dim(str(int(size))+'MB'))
   idx+=1
  print()
 flat=[p for dd,p in files]
 print('  [nomor] hapus file (0 / 0,2 / 0-3)   [F] isi folder   [A] SEMUA   [Q] batal')
 print()
 c=input('  > ').strip().upper()
 if c=='Q':return
 if c=='F':
  print()
  for i,d in enumerate(roots):print('  ['+str(i)+'] '+str(d))
  print()
  v=input('  Folder: ').strip()
  try:dd=roots[int(v)]
  except:return
  go=input('  Ketik YA untuk hapus semua isi '+str(dd)+': ').strip()
  if go=='YA':
   shutil.rmtree(dd,ignore_errors=True)
   dd.mkdir(parents=True,exist_ok=True)
   print(ok('  Folder dikosongkan.'))
  input('\n  Enter...');return
 if c=='A':
  go=input('  Ketik HAPUS untuk hapus SEMUA file di 4 folder: ').strip()
  if go=='HAPUS':
   n=0
   for p in flat:
    try:os.remove(p);n+=1
    except:pass
   print(ok('  '+str(n)+' file dihapus.'))
  input('\n  Enter...');return
 try:
  nums=[]
  for part in c.split(','):
   part=part.strip()
   if '-' in part:
    x,y=part.split('-',1);nums.extend(range(int(x),int(y)+1))
   elif part.isdigit():nums.append(int(part))
  sel=[flat[n] for n in nums if 0<=n<len(flat)]
  if not sel:return
  tot=sum(p.stat().st_size for p in sel)/1024/1024
  print('\n  Hapus '+str(len(sel))+' file ('+str(round(tot,1))+' MB)?')
  for p in sel:print('    - '+p.name)
  go=input('  Ketik Y untuk lanjut: ').strip().upper()
  if go=='Y':
   for p in sel:
    try:os.remove(p)
    except:pass
   print(ok('  Dihapus.'))
 except:pass
 input('\n  Enter...')

def menu_browse():
 ci();hdr('BROWSE FILES')
 dirs=[UPLOAD,OUTPUT,Path('/content')]
 print()
 print('  [1] '+str(UPLOAD))
 print('  [2] '+str(OUTPUT))
 print('  [3] /content/')
 print()
 print('  [0] Kembali')
 print()
 c=input('  Pilih: ').strip()
 if c=='0':return
 try:
  d=dirs[int(c)-1]
 except:return
 if not d.exists():print(er('  Folder tidak ada.'));input('  Enter...');return
 print()
 subprocess.run(['tree','--dirsfirst','-L','2',str(d)])
 print()
 input('  Enter...')

def main():
 load_secrets()
 while True:
  ci()
  print('\n'+'\033[96m'+'='*62+'\033[0m')
  print('\033[96m  haru-mux v2026.09.08b -- MKV Muxing Tool\033[0m')
  print('\033[96m'+'='*62+'\033[0m')
  print()
  print('  [1]  Download       -- Gofile / GDrive / URL')
  print('  [2]  Mux            -- Pilih file, edit track, mux')
  print('  [3]  List Tracks    -- Lihat semua track di file')
  print('  [4]  MediaInfo      -- Cek info media file')
  print('  [5]  Upload         -- Upload hasil muxing')
  print('  [6]  Browse         -- Lihat isi folder')
  print('  [7]  Batch series     -- Pair sub dengan video per episode')
  print('  [8]  Hapus file         -- File manager (satuan/folder/semua)')
  print()
  print('  [Q]  Keluar')
  secs=[]
  if get_secret('GOFILE_API_TOKEN'):secs.append('gofile')
  if get_secret('GDRIVE_REFRESH_TOKEN'):secs.append('gdrive')
  if get_secret('OWNER_ID') and get_secret('HARU_BOT_TOKEN'):secs.append('telegram')
  print()
  print('  Secrets: '+(dim(', '.join(secs)) if secs else er('KOSONG! re-run cell Install')))
  print()
  c=input('  Pilih: ').strip().upper()
  if c=='Q':print('\n  Bye!');sys.exit(0)
  elif c=='1':menu_download()
  elif c=='2':menu_mux()
  elif c=='3':menu_list()
  elif c=='4':menu_info()
  elif c=='5':menu_upload()
  elif c=='6':menu_browse()
  elif c=='7':menu_batch()
  elif c=='8':menu_delete()

if __name__=='__main__':main()'''
script_path = '/usr/local/bin/haru-mux'
with open(script_path, 'w') as f:
    f.write(HARU_MUX_SCRIPT)
os.chmod(script_path, 0o755)
print('HARU-EXTRACT installing...')
HARU_EXTRACT_SCRIPT = r'''#!/usr/bin/env python3
import subprocess,sys,os,re,glob,json,time
import requests
from pathlib import Path
V={'.mkv','.mp4','.avi','.mov','.webm','.flv','.wmv','.ts','.m4v'}
A={'.mp3','.aac','.flac','.wav','.ogg','.opus','.mka','.ac3','.dts','.eac3','.m4a'}
S={'.srt','.ass','.ssa','.sub','.idx','.sup','.vtt','.pgs','.scc','.sami'}
L={'id':'Indonesian','en':'English','ja':'Japanese','ko':'Korean','zh':'Chinese','ms':'Malay','ar':'Arabic','de':'German','fr':'French','es':'Spanish','pt':'Portuguese','ru':'Russian','it':'Italian','th':'Thai','vi':'Vietnamese','hi':'Hindi','und':'Undetermined'}
UPLOAD=Path('/content/uploads')
OUTPUT=Path('/content/output')
EXTDIR=Path('/content/extracts')
EXTDIR.mkdir(exist_ok=True)
TGBOT=''
def tg_owner():
 return get_secret('OWNER_ID')
def tg_token():
 return get_secret('HARU_BOT_TOKEN')
def tg_send(msg):
 oid=tg_owner()
 tok=tg_token()
 if not oid or not tok:return
 try:requests.post('https://api.telegram.org/bot'+tok+'/sendMessage',json={'chat_id':oid,'text':msg,'parse_mode':'HTML','disable_web_page_preview':True},timeout=10)
 except:pass
def ci():os.system('cls' if os.name=='nt' else 'clear')
def ok(t):return '\033[92m'+t+'\033[0m'
def er(t):return '\033[91m'+t+'\033[0m'
def dim(t):return '\033[90m'+t+'\033[0m'
def hdr(title):print('\n'+'='*62);print('  '+title);print('='*62)
def auto_lang(fn):
 fn=fn.lower()
 for k,c in {'[id]':'id','indonesian':'id','indo':'id','[en]':'en','english':'en','[ja]':'ja','japanese':'ja','jpn':'ja','[ko]':'ko','[zh]':'zh'}.items():
  if k in fn:return c
 return 'und'
def norm_lang(code):
 code=str(code or '').strip().lower()
 m3={'jpn':'ja','eng':'en','ind':'id','kor':'ko','chi':'zh','zho':'zh','msa':'ms','ara':'ar','ger':'de','deu':'de','fre':'fr','fra':'fr','spa':'es','por':'pt','rus':'ru','ita':'it','tha':'th','vie':'vi','hin':'hi','und':'und'}
 if code in m3:return m3[code]
 full={'japanese':'ja','english':'en','indonesian':'id','korean':'ko','chinese':'zh','malay':'ms','arabic':'ar','german':'de','french':'fr','spanish':'es','portuguese':'pt','russian':'ru','italian':'it','thai':'th','vietnamese':'vi','hindi':'hi'}
 if code in full:return full[code]
 if code in L:return code
 return code if code else 'und'
def _det_type(f):
 e=Path(f).suffix.lower()
 if e in V:return 'video'
 if e in A:return 'audio'
 if e in S:return 'subtitle'
 return 'other'
def _pad(s,w):
 s=str(s)
 if len(s)>w:return s[:w-2]+'..'
 return s+(' '*(w-len(s)))
def codec_ext(codec,ttype):
 c=codec.lower()
 tab=[('opus','opus'),('aac','aac'),('e-ac-3','eac3'),('ac-3','ac3'),('ac3','ac3'),('dts','dts'),('flac','flac'),('mp3','mp3'),('vorbis','ogg'),('pcm','wav'),('substation','ass'),('ass','ass'),('subrip','srt'),('srt','srt'),('pgs','sup'),('vobsub','sub'),('dvbsub','sub'),('av1','ivf'),('vp9','ivf'),('avc','h264'),('hevc','h265'),('mpeg','mpg')]
 for k,e in tab:
  if k in c:return e
 if ttype=='audio':return 'mka'
 if ttype=='subtitle':return 'srt'
 return 'bin'
def probe_file(f):
 f=Path(f)
 tracks=[]
 try:
  r=subprocess.run(['mkvmerge','-J',str(f)],capture_output=True,text=True,timeout=30)
  if r.returncode==0 and r.stdout.strip():
   data=json.loads(r.stdout)
   nchap=len(data.get('chapters',[]))
   for tr in data.get('tracks',[]):
    ttype=str(tr.get('type','')).lower()
    if ttype=='subtitles':ttype='subtitle'
    props=tr.get('properties',{}) or {}
    lang=norm_lang(props.get('language','und'))
    if lang=='und':lang=auto_lang(f.name)
    tracks.append({'file':str(f),'file_name':f.name,'track_id':int(tr.get('id',0)),'codec':str(tr.get('codec','')),'type':ttype,'language':lang,'name':str(props.get('track_name','') or ''),'chapters':nchap})
   if tracks:return tracks
 except:pass
 return tracks
def scan_sources():
 fs=[]
 for d in [UPLOAD,OUTPUT,EXTDIR]:
  if not d.exists():continue
  for p in sorted(d.rglob('*')):
   if p.is_file() and p.suffix.lower() in V:fs.append(p)
 return fs
def sel_sources():
 ci();hdr('PILIH FILE SUMBER')
 fs=scan_sources()
 if not fs:print('\n  '+er('Tidak ada video di uploads/output/extracts.'));input('  Enter...');return None
 print()
 for i,f in enumerate(fs):
  size=f.stat().st_size/1024/1024
  print('  ['+str(i)+'] '+f.name+'  '+dim(str(int(size))+'MB'))
 print()
 print('  '+'-'*50)
 print('  Pilih: 0  atau  0,1  atau  * (semua)')
 print('  '+'-'*50)
 print()
 while True:
  c=input('  > ').strip()
  if not c:continue
  if c=='*':return fs
  try:
   nums=[]
   for part in c.split(','):
    part=part.strip()
    if '-' in part:
     a,b=part.split('-',1);nums.extend(range(int(a),int(b)+1))
    else:nums.append(int(part))
   sel=[fs[n] for n in nums if 0<=n<len(fs)]
   if sel:return sel
  except:print('  Input tidak valid!')
def show_tracks(all_tracks):
 print()
 print('  '+_pad('No',2)+'  '+_pad('Codec',20)+'  '+_pad('Type',8)+'  '+_pad('Lang',4)+'  '+_pad('Name',30)+'  '+_pad('TID',3))
 print('  '+'-'*62)
 by_file={}
 for t in all_tracks:by_file.setdefault(t['file'],[]).append(t)
 for filepath,tracks in by_file.items():
  ch=tracks[0].get('chapters',0)
  chs='  '+str(ch)+' chapters' if ch else ''
  print('  [V] '+tracks[0]['file_name']+' ('+str(len(tracks))+' tracks'+chs+')')
  for t in tracks:
   nm=_pad(t['name'] if t['name'] else '-',18)
   print('  '+_pad(t['global_idx'],2)+'  '+_pad(t['codec'],20)+'  '+_pad(t['type'],8)+'  '+_pad(t['language'],4)+'  '+nm+'  '+_pad(t['track_id'],3))
  print()
def load_all(srcs):
 all_tracks=[]
 for fp in srcs:
  for x in probe_file(fp):all_tracks.append(x)
 for i,t in enumerate(all_tracks):t['global_idx']=i
 return all_tracks
def menu_extract():
 srcs=sel_sources()
 if not srcs:return
 all_tracks=load_all(srcs)
 if not all_tracks:print(er('  Tidak ada track.'));input('  Enter...');return
 ci();hdr('PILIH TRACK')
 show_tracks(all_tracks)
 print('  [1] Semua audio     [2] Semua subtitle   [3] Semua video')
 print('  [4] Track pilihan (0,2 / 0-3)   [5] Semua track')
 print('  [0] Kembali')
 print()
 c=input('  > ').strip()
 if c=='0':return
 elif c=='1':jobs=[t for t in all_tracks if t['type']=='audio']
 elif c=='2':jobs=[t for t in all_tracks if t['type']=='subtitle']
 elif c=='3':jobs=[t for t in all_tracks if t['type']=='video']
 elif c=='5':jobs=list(all_tracks)
 elif c=='4':
  s=input('  Nomor track (0,2 / 0-3): ').strip()
  try:
   nums=[]
   for part in s.split(','):
    part=part.strip()
    if '-' in part:
     a,b=part.split('-',1);nums.extend(range(int(a),int(b)+1))
    else:nums.append(int(part))
   want=set(nums)
   jobs=[t for t in all_tracks if t['global_idx'] in want]
  except:print(er('  Input tidak valid.'));return
 else:return
 if not jobs:print(er('  Tidak ada track cocok.'));input('  Enter...');return
 pre={'audio':'aud','subtitle':'sub','video':'vid'}
 by_src={}
 for t in jobs:by_src.setdefault(t['file'],[]).append(t)
 total_ok=0
 for srcpath,tracks in by_src.items():
  stem=Path(srcpath).stem
  args=[]
  for t in tracks:
   ext=codec_ext(t['codec'],t['type'])
   base='['+pre.get(t['type'],'trk')+'_'+t['language']+'] '+stem+'.'+ext
   out=EXTDIR/base;n=2
   while out.exists():out=EXTDIR/('['+pre.get(t['type'],'trk')+'_'+t['language']+'] '+stem+'_'+str(n)+'.'+ext);n+=1
   args.append(str(t['track_id'])+':'+str(out))
   t['_out']=str(out)
  print('\n  Extract dari '+Path(srcpath).name+' ('+str(len(tracks))+' track)...')
  r=subprocess.run(['mkvextract','tracks',srcpath]+args,capture_output=True,text=True,timeout=600)
  for t in tracks:
   p=Path(t['_out'])
   if p.exists() and p.stat().st_size>0:
    mb=p.stat().st_size/1024/1024
    print('  '+ok('OK')+' '+p.name+' ('+str(round(mb,1))+' MB)')
    total_ok+=1
   else:print('  '+er('GAGAL')+' TID '+str(t['track_id'])+' '+r.stderr[-200:])
 xnames=[]
 for t in jobs:
  o=t.get('_out','')
  if o and Path(o).exists():xnames.append(Path(o).name)
 print('\n  Selesai: '+str(total_ok)+'/'+str(len(jobs))+' track -> '+str(EXTDIR))
 xmsg='<b>Extract selesai</b>\n'+str(total_ok)+'/'+str(len(jobs))+' track'
 if xnames:xmsg=xmsg+'\n'+'\n'.join(xnames[:20])
 if len(xnames)>20:xmsg=xmsg+'\n... +'+str(len(xnames)-20)+' lagi'
 tg_send(xmsg)
 input('\n  Enter...')
def menu_list():
 srcs=sel_sources()
 if not srcs:return
 all_tracks=load_all(srcs)
 if not all_tracks:print(er('  Tidak ada track.'));input('  Enter...');return
 ci();hdr('LIST TRACKS')
 show_tracks(all_tracks)
 input('  Enter...')
def load_secrets():
 try:
  if os.path.exists('/content/.haru_secrets.json'):
   d=json.load(open('/content/.haru_secrets.json'))
   for k,v in d.items():
    if v and not os.environ.get(k):os.environ[k]=str(v)
 except:pass
def get_secret(k):
 v=os.environ.get(k,'')
 if v:return v.strip()
 try:
  from google.colab import userdata
  t=userdata.get(k)
  if t:return str(t).strip()
 except:pass
 return ''
def get_gofile_token():
 return get_secret('GOFILE_API_TOKEN')
def gofile_wt(agent,token):
 import hashlib,time
 slot=int(time.time())//14400
 return hashlib.sha256((agent+'::en-US::'+token+'::'+str(slot)+'::12af056dacea0b').encode()).hexdigest()

def gofile_direct_fetch(url,password):
 import hashlib
 m=re.search(r'gofile\.io/d/(\w+)',url)
 if not m:return None,'Link tidak valid',None
 cid=m.group(1)
 pw=hashlib.sha256(password.encode()).hexdigest() if password else None
 agent='Mozilla/5.0'
 s=requests.Session()
 s.headers.update({'Accept-Encoding':'gzip','User-Agent':agent,'Connection':'keep-alive','Accept':'*/*','Origin':'https://gofile.io','Referer':'https://gofile.io/'})
 try:
  r=s.post('https://api.gofile.io/accounts',headers={'X-Website-Token':gofile_wt(agent,''),'X-BL':'en-US'},timeout=20)
  tok=r.json()['data']['token']
 except Exception as e:return None,'Guest account gagal: '+str(e)[:120],None
 s.cookies.set('Cookie','accountToken='+tok)
 s.headers.update({'Authorization':'Bearer '+tok})
 files=[]
 try:
  def walk(x):
   u='https://api.gofile.io/contents/'+x+'?cache=true'
   if pw:u=u+'&password='+pw
   r=s.get(u,headers={'X-Website-Token':gofile_wt(agent,tok),'X-BL':'en-US'},timeout=30)
   d=r.json()
   if d.get('status')!='ok':raise Exception(str(d.get('status'))[:60])
   data=d['data']
   if data.get('passwordStatus','passwordOk')!='passwordOk' and 'password' in data:raise Exception('password salah')
   if data.get('type')!='folder':
    if data.get('link'):files.append({'name':data['name'],'size':data.get('size',0),'link':data['link']})
    return
   for ch in (data.get('children',{}) or {}).values():
    if ch.get('type')=='folder':walk(ch['id'])
    elif ch.get('link'):files.append({'name':ch['name'],'size':ch.get('size',0),'link':ch['link']})
  walk(cid)
 except Exception as e:return None,'List gagal: '+str(e)[:150],None
 return files,None,tok

def gofile_direct_one(f,tok,dest_dir):
 name=f['name'];dest=dest_dir/name;part=dest_dir/(name+'.part')
 if dest.exists() and dest.stat().st_size>0:
  print('  SKIP '+name+' (sudah ada)');return True
 hdr={'User-Agent':'Mozilla/5.0','Referer':'https://gofile.io/','Origin':'https://gofile.io','Cookie':'accountToken='+tok}
 for att in range(1,4):
  try:
   print('  Direct '+name+'...'+('' if att==1 else ' (coba '+str(att)+')'))
   rr=requests.get(f['link'],headers=hdr,stream=True,timeout=600)
   rr.raise_for_status()
   total=0
   fh=open(part,'wb')
   for ch in rr.iter_content(chunk_size=1024*1024):
    if ch:fh.write(ch);total+=len(ch)
   fh.close()
   if total==0:raise Exception('0 byte')
   os.rename(part,dest)
   print('  OK '+name+' ('+str(total)+' bytes / '+str(round(total/1024/1024,1))+' MB)')
   return True
  except Exception as e:
   try:fh.close()
   except:pass
   try:
    if part.exists():os.remove(part)
   except:pass
   if att<3:
    print('  Gagal, retry... ('+str(e)[:120]+')')
    time.sleep(10*att)
   else:print(er('  Gagal: '+name+' - '+str(e)[:150]))
 return False

def gofile_direct_retry(url,pwd,names,dest_dir):
 print('  Coba jalur direct API untuk '+str(len(names))+' file...')
 files,err,tok=gofile_direct_fetch(url,pwd)
 if err:print(er('  Direct: '+err));return names
 targets=[f for f in files if f['name'] in names]
 if not targets:print(er('  Direct: file tidak ketemu di listing.'));return names
 still=[]
 for f in targets:
  if not gofile_direct_one(f,tok,dest_dir):still.append(f['name'])
 return still


def dl_gofile():
 print('  Redirecting ke haru-download...');subprocess.run(['haru-download'])

def dl_gofile():
 print('  Redirecting ke haru-download...');subprocess.run(['haru-download'])
def dl_drive():dl_gofile()

def dl_gofile():
 print('  Redirecting ke haru-download...');subprocess.run(['haru-download'])
def dl_drive():dl_gofile()
def dl_url():dl_gofile()

def dl_gofile():
 print('  Redirecting ke haru-download...');subprocess.run(['haru-download'])
def dl_drive():dl_gofile()
def dl_url():dl_gofile()
def menu_download():dl_gofile()

def menu_download():dl_gofile()

def dl_url():dl_gofile()
def menu_download():dl_gofile()

def dl_drive():
 hdr('DOWNLOAD - Google Drive')
 url=input('\n  Link/folder GDrive: ').strip()
 if not url:return
 sub=input('  Subfolder di extracts/ (kosong = langsung): ').strip()
 dest=EXTDIR/sub if sub else EXTDIR
 dest.mkdir(parents=True,exist_ok=True)
 print('  Downloading ke '+str(dest)+'...')
 subprocess.run(['gdown','--folder','-O',str(dest),'--remaining-ok',url],timeout=600)
 print(ok('  Selesai!'))
def dl_url():
 hdr('DOWNLOAD - Direct URL')
 url=input('\n  Direct URL: ').strip()
 if not url:return
 fname=input('  Filename (kosong = auto): ').strip() or None
 cmd=['wget','-q','-P',str(EXTDIR),'--content-disposition','--no-check-certificate']
 if fname:cmd.extend(['-O',str(EXTDIR/fname)])
 cmd.append(url)
 subprocess.run(cmd,timeout=600)
 print(ok('  Selesai!'))
def menu_download():
 while True:
  ci();hdr('DOWNLOAD')
  print()
  print('  [1] Gofile')
  print('  [2] Google Drive')
  print('  [3] Direct URL')
  print()
  print('  [0] Kembali')
  print()
  c=input('  Pilih: ').strip()
  if c=='0':return
  elif c=='1':dl_gofile()
  elif c=='2':dl_drive()
  elif c=='3':dl_url()
  input('\n  Enter...')
def gdrive_secret(k):
 return get_secret(k)
def upload_gofile():
 print('  Redirecting ke haru-upload...');subprocess.run(['haru-upload'])

def upload_gofile():
 print('  Redirecting ke haru-upload...');subprocess.run(['haru-upload'])
def upload_drive():upload_gofile()

def upload_gofile():
 print('  Redirecting ke haru-upload...');subprocess.run(['haru-upload'])
def upload_drive():upload_gofile()
def menu_upload():upload_gofile()

def menu_upload():upload_gofile()

def upload_drive():
 hdr('UPLOAD - Google Drive')
 all_files=[]
 for d in [UPLOAD,OUTPUT,Path('/content/extracts'),Path('/content/downloads')]:
  if d.exists():
   for f in sorted(d.rglob('*')):
    if f.is_file() and f.suffix.lower() in V|A|S:all_files.append((d,f))
 if not all_files:print(er('  Tidak ada file untuk di-upload.'));input('  Enter...');return
 print()
 idx=0
 for d in [UPLOAD,OUTPUT,Path('/content/extracts'),Path('/content/downloads')]:
  grp=[(dd,f) for dd,f in all_files if dd==d]
  if not grp:continue
  print('  ['+d.name+'/]  ('+str(len(grp))+' file)')
  for dd,f in grp:
   size=f.stat().st_size/1024/1024
   print('    ['+str(idx)+'] '+f.name+'  '+dim(str(int(size))+'MB'))
   idx+=1
  print()
 flat=[f for dd,f in all_files]
 c=input('  Pilih (* semua / 0,1,2 / 0-3 / Q batal): ').strip().upper()
 if c=='Q':return
 if c=='*':targets=flat
 else:
  try:
   nums=[]
   for part in c.split(','):
    part=part.strip()
    if '-' in part:a,b=part.split('-',1);nums.extend(range(int(a),int(b)+1))
    else:nums.append(int(part))
   targets=[flat[n] for n in nums if 0<=n<len(flat)]
  except:print('  Input tidak valid.');input('  Enter...');return
  if not targets:return
 cid=gdrive_secret('GDRIVE_CLIENT_ID');sec=gdrive_secret('GDRIVE_CLIENT_SECRET');ref=gdrive_secret('GDRIVE_REFRESH_TOKEN')
 parent_id=gdrive_secret('GDRIVE_FOLDER_ID') or '1pjpd63PTFvwYd8iI7dvMwcU-e_LMqvUE'
 if not(cid and sec and ref):
  print(er('  Secret GDrive tidak kebaca.'));print('  Aktifkan toggle secret + re-run cell Install.');input('  Enter...');return
 print('  Auth via API...')
 tok=gdrive_token(cid,sec,ref)
 if not tok:print(er('  Gagal dapat access token.'));return
 import re
 m=re.search(r'/folders/([A-Za-z0-9_-]+)',parent_id)
 if m:parent_id=m.group(1)
 elif len(parent_id)<20:
  q="name='"+parent_id+"' and mimeType='application/vnd.google-apps.folder' and trashed=false"
  try:
   r=requests.get('https://www.googleapis.com/drive/v3/files',headers={'Authorization':'Bearer '+tok},params={'q':q,'fields':'files(id)'},timeout=15)
   fs=r.json().get('files',[])
   if fs:parent_id=fs[0]['id']
  except:pass
 sub=input('  Subfolder ['+dim('langsung ke parent')+']: ').strip()
 target=parent_id
 if sub:
  try:
   q2="name='"+sub+"' and '"+parent_id+"' in parents and mimeType='application/vnd.google-apps.folder' and trashed=false"
   r2=requests.get('https://www.googleapis.com/drive/v3/files',headers={'Authorization':'Bearer '+tok},params={'q':q2,'fields':'files(id)'},timeout=15)
   fs2=r2.json().get('files',[])
   if fs2:target=fs2[0]['id']
   else:
    meta={'name':sub,'mimeType':'application/vnd.google-apps.folder','parents':[parent_id]}
    r3=requests.post('https://www.googleapis.com/drive/v3/files',headers={'Authorization':'Bearer '+tok,'Content-Type':'application/json'},data=json.dumps(meta),timeout=15)
    nid=r3.json().get('id')
    if nid:target=nid;print('  Subfolder dibuat: '+sub)
    else:print(er('  Gagal buat subfolder.'))
  except:print(er('  Error buat subfolder.'))
 ok_n=0;fail=[]
 for f in targets:
  print('  Upload '+f.name+' ('+str(round(f.stat().st_size/1024/1024,1))+'MB)...')
  if gdrive_upload_file(tok,f,target):ok_n+=1;print('  '+ok('ok')+' '+f.name)
  else:fail.append(f.name);print('  '+er('gagal')+' '+f.name)
 if ok_n:tg_send('<b>Upload GDrive</b>\n'+str(ok_n)+' file berhasil')
 if fail:print(er('  Gagal: '+', '.join(fail)))
 input('\n  Enter...')


def page_out(text):
 ls=text.splitlines()
 if len(ls)>80:
  import tempfile
  tp=os.path.join(tempfile.gettempdir(),'mi.txt')
  open(tp,'w',encoding='utf-8',errors='ignore').write(text)
  print('  Output panjang ('+str(len(ls))+' baris) -> less, q keluar, panah scroll')
  try:subprocess.run(['less','-M','-X',tp])
  except:print(text)
 else:print(text)

def telegraph_upload(title,text):
 try:
  r=requests.post('https://api.telegra.ph/createAccount',data={'short_name':'haru','author_name':'haru-extract'},timeout=20)
  tok=r.json()['result']['access_token']
 except:return None
 try:
  nodes=json.dumps([{'tag':'pre','children':[text[:60000]]}])
  r=requests.post('https://api.telegra.ph/createPage',data={'access_token':tok,'title':title[:60],'author_name':'haru-extract','content':nodes},timeout=30)
  d=r.json()
  if d.get('ok'):print(ok('  '+d['result']['url']));return d['result']['url']
 except:pass
 return None

def telegraph_bulk(title,sections,author):
 pages=[];cur=[];curlen=0
 for name,text in sections:
  bl=len(name)+len(text)+100
  if cur and curlen+bl>58000:
   pages.append(cur);cur=[];curlen=0
  cur.append((name,text));curlen+=bl
 if cur:pages.append(cur)
 urls=[]
 for i,pg in enumerate(pages):
  nodes=[]
  for name,text in pg:
   nodes.append({'tag':'h4','children':[name]})
   nodes.append({'tag':'pre','children':[text[:60000]]})
  try:
   r=requests.post('https://api.telegra.ph/createAccount',data={'short_name':'haru','author_name':haru-extract},timeout=20)
   tok=r.json()['result']['access_token']
   t=title+(' (%d/%d)'%(i+1,len(pages)) if len(pages)>1 else '')
   r=requests.post('https://api.telegra.ph/createPage',data={'access_token':tok,'title':t[:60],'author_name':haru-extract,'content':json.dumps(nodes)},timeout=30)
   d=r.json()
   if d.get('ok'):urls.append(d['result']['url']);print(ok('  Hal '+str(i+1)+': '+d['result']['url']))
  except Exception as e:print(er('  Gagal hal '+str(i+1)))
 return urls

def menu_info():
 ci();hdr('MEDIAINFO')
 print()
 print('  [1] Pilih file (satuan/*)')
 print('  [2] Bulk 1 folder -> telegra.ph gabungan')
 print()
 print('  [0] Kembali')
 print()
 c=input('  Pilih: ').strip()
 if c=='0':return
 if c=='2':return mi_bulk()
 items=[]
 for d in [UPLOAD,OUTPUT,EXTDIR]:
  if d.exists():
   for f in sorted(d.rglob('*')):
    if f.is_file() and f.suffix.lower() in V|A|S:items.append((d,f))
 if not items:print(er('  Tidak ada file.'));input('  Enter...');return
 print()
 idx=0
 for d in [UPLOAD,OUTPUT,EXTDIR]:
  grp=[f for dd,f in items if dd==d]
  if not grp:continue
  print('  ['+d.name+'/]')
  for f in grp:
   size=f.stat().st_size/1024/1024
   print('  ['+str(idx)+'] '+f.name+'  '+dim(str(int(size))+'MB'))
   idx+=1
  print()
 flat=[f for dd,f in items]
 c=input('  Pilih file (atau * semua): ').strip()
 if c=='*':targets=flat
 else:
  try:
   idx=int(c)
   if 0<=idx<len(flat):targets=[flat[idx]]
   else:return
  except:return
 fmt=input('  Format (T=text, J=json) [T]: ').strip().upper() or 'T'
 saved=[]
 for f in targets:
  cmd=['mediainfo']
  if fmt=='J':cmd.append('--Output=JSON')
  cmd.append(str(f))
  r=subprocess.run(cmd,capture_output=True,text=True,timeout=30)
  page_out(r.stdout)
  saved.append((f.name,r.stdout))
 if saved:
  u=input('\n  Upload ke telegra.ph? [Y/n]: ').strip().lower()
  if u in ('','y'):
   links=[]
   for name,text in saved:
    url=telegraph_upload('MediaInfo - '+name,text)
    if url:links.append((name,url))
   if links:
    msg='<b>MediaInfo</b>'
    for name,url in links:msg=msg+'\n'+name+'\n'+url
    tg_send(msg)
 input('  Enter...')

def mi_bulk():
 ci();hdr('BULK MEDIAINFO')
 dirs=[d for d in [UPLOAD,OUTPUT,EXTDIR] if d.exists()]
 if not dirs:return
 print()
 for i,d in enumerate(dirs):print('  ['+str(i)+'] '+str(d))
 print()
 c=input('  Folder: ').strip()
 try:d=dirs[int(c)]
 except:return
 fs=[p for p in sorted(d.rglob('*')) if p.is_file() and p.suffix.lower() in V|A|S]
 if not fs:print(er('  Kosong.'));input('  Enter...');return
 print('\n  Proses '+str(len(fs))+' file...')
 sections=[]
 for f in fs:
  r=subprocess.run(['mediainfo',str(f)],capture_output=True,text=True,timeout=30)
  sections.append((f.name,r.stdout))
  print('  ok '+f.name)
 print()
 urls=telegraph_bulk('MediaInfo - '+d.name+' ('+str(len(fs))+' file)',sections,'haru-extract')
 if urls:
  msg='<b>Bulk MediaInfo</b>\n'+str(len(fs))+' file'
  for u in urls:msg=msg+'\n'+u
  tg_send(msg)
 input('\n  Enter...')


def menu_upload():
 ci();hdr('UPLOAD')
 print()
 print('  [1] Gofile  (folder gabungan)')
 print('  [2] Google Drive (multi-file + subfolder)')
 print()
 print('  [0] Kembali')
 print()
 c=input('  Pilih: ').strip()
 if c=='0':return
 elif c=='1':upload_gofile()
 elif c=='2':upload_drive()


def menu_browse():
 ci();hdr('BROWSE')
 print()
 subprocess.run(['tree','--dirsfirst','-L','3',str(EXTDIR)])
 print()
 input('  Enter...')
def main():
 load_secrets()
 while True:
  ci()
  print('\n'+'\033[96m'+'='*62+'\033[0m')
  print('\033[96m  haru-extract v2026.09.08b -- Track Extractor\033[0m')
  print('\033[96m'+'='*62+'\033[0m')
  print()
  print('  [1]  Download       -- Gofile / GDrive / URL')
  print('  [2]  Extract        -- Pilih file, pilih track, extract')
  print('  [3]  List Tracks    -- Lihat semua track')
  print('  [4]  MediaInfo      -- Satuan / bulk -> telegra.ph')
  print('  [5]  Upload         -- Upload hasil extract')
  print('  [6]  Browse         -- Lihat isi extracts/')
  print()
  print('  [Q]  Keluar')
  secs=[]
  if get_secret('GOFILE_API_TOKEN'):secs.append('gofile')
  if get_secret('GDRIVE_REFRESH_TOKEN'):secs.append('gdrive')
  if get_secret('OWNER_ID') and get_secret('HARU_BOT_TOKEN'):secs.append('telegram')
  print()
  print('  Secrets: '+(dim(', '.join(secs)) if secs else er('KOSONG! re-run cell Install')))
  print()
  c=input('  Pilih: ').strip().upper()
  if c=='Q':print('\n  Bye!');sys.exit(0)
  elif c=='1':menu_download()
  elif c=='2':menu_extract()
  elif c=='3':menu_list()
  elif c=='4':menu_info()
  elif c=='5':menu_upload()
  elif c=='6':menu_browse()
if __name__=='__main__':main()'''
extract_path = '/usr/local/bin/haru-extract'
with open(extract_path, 'w') as f:
    f.write(HARU_EXTRACT_SCRIPT)
os.chmod(extract_path, 0o755)
print('HARU-METADATA installing...')
HARU_META_SCRIPT = r'''#!/usr/bin/env python3
import subprocess,sys,os,re,glob,json,time
import requests
from pathlib import Path
V={'.mkv','.mp4','.avi','.mov','.webm','.flv','.wmv','.ts','.m4v'}
MKVOK={'.mkv','.mka','.mks','.webm'}
A={'.mp3','.aac','.flac','.wav','.ogg','.opus','.mka','.ac3','.dts','.eac3','.m4a'}
S={'.srt','.ass','.ssa','.sub','.idx','.sup','.vtt','.pgs','.scc','.sami'}
L={'id':'Indonesian','en':'English','ja':'Japanese','ko':'Korean','zh':'Chinese','ms':'Malay','ar':'Arabic','de':'German','fr':'French','es':'Spanish','pt':'Portuguese','ru':'Russian','it':'Italian','th':'Thai','vi':'Vietnamese','hi':'Hindi','und':'Undetermined'}
UPLOAD=Path('/content/uploads')
OUTPUT=Path('/content/output')
OUTPUT.mkdir(exist_ok=True)
def ci():os.system('cls' if os.name=='nt' else 'clear')
def ok(t):return '\033[92m'+t+'\033[0m'
def er(t):return '\033[91m'+t+'\033[0m'
def dim(t):return '\033[90m'+t+'\033[0m'
def hdr(title):print('\n'+'='*62);print('  '+title);print('='*62)
def _pad(s,w):
 s=str(s)
 if len(s)>w:return s[:w-2]+'..'
 return s+(' '*(w-len(s)))
def load_secrets():
 try:
  if os.path.exists('/content/.haru_secrets.json'):
   d=json.load(open('/content/.haru_secrets.json'))
   for k,v in d.items():
    if v and not os.environ.get(k):os.environ[k]=str(v)
 except:pass
def get_secret(k):
 v=os.environ.get(k,'')
 if v:return v.strip()
 try:
  from google.colab import userdata
  t=userdata.get(k)
  if t:return str(t).strip()
 except:pass
 return ''
def norm_lang(code):
 code=str(code or '').strip().lower()
 m3={'jpn':'ja','eng':'en','ind':'id','kor':'ko','chi':'zh','zho':'zh','msa':'ms','ara':'ar','ger':'de','deu':'de','fre':'fr','fra':'fr','spa':'es','por':'pt','rus':'ru','ita':'it','tha':'th','vie':'vi','hin':'hi','und':'und'}
 if code in m3:return m3[code]
 full={'japanese':'ja','english':'en','indonesian':'id','korean':'ko','chinese':'zh','malay':'ms','arabic':'ar','german':'de','french':'fr','spanish':'es','portuguese':'pt','russian':'ru','italian':'it','thai':'th','vietnamese':'vi','hindi':'hi'}
 if code in full:return full[code]
 if code in L:return code
 return code if code else 'und'
def tg_owner():
 return get_secret('OWNER_ID')
def tg_token():
 return get_secret('HARU_BOT_TOKEN')
def tg_send(msg):
 oid=tg_owner();tok=tg_token()
 if not oid or not tok:return
 try:requests.post('https://api.telegram.org/bot'+tok+'/sendMessage',json={'chat_id':oid,'text':msg,'parse_mode':'HTML','disable_web_page_preview':True},timeout=10)
 except:pass
def probe_meta(f):
 f=Path(f)
 tracks=[];title='';chapters=[]
 try:
  r=subprocess.run(['mkvmerge','-J',str(f)],capture_output=True,text=True,timeout=30)
  if r.returncode==0 and r.stdout.strip():
   data=json.loads(r.stdout)
   cprops=(data.get('container',{}) or {}).get('properties',{}) or {}
   title=str(cprops.get('title','') or '')
   for tr in data.get('tracks',[]):
    ttype=str(tr.get('type','')).lower()
    if ttype=='subtitles':ttype='subtitle'
    props=tr.get('properties',{}) or {}
    tracks.append({'track_id':int(tr.get('id',0)),'uid':props.get('uid',0),'codec':str(tr.get('codec','')),'type':ttype,'language':norm_lang(props.get('language','und')),'name':str(props.get('track_name','') or ''),'default':'yes' if props.get('default_track',False) else 'no','forced':'yes' if props.get('forced_track',False) else 'no','enabled':'yes' if props.get('enabled_track',True) else 'no'})
   for i,ch in enumerate(data.get('chapters',[])):
    nm=ch.get('name') or ''
    if not nm:
     for k in ('chapter_string','string','title'):
      if ch.get(k):nm=str(ch[k]);break
    chapters.append({'no':i+1,'name':nm or ('Chapter '+str(i+1)),'start':ch.get('time_start',0)})
 except Exception as e:print(er('  Probe gagal: '+str(e)[:150]))
 return tracks,title,chapters
def show_tracks(ts):
 print()
 print('  '+_pad('No',2)+'  '+_pad('Codec',20)+'  '+_pad('Type',8)+'  '+_pad('Lang',4)+'  '+_pad('Name',30)+'  '+_pad('TID',3)+'  Def  Forced En')
 print('  '+'-'*80)
 for i,t in enumerate(ts):
  de=ok('Yes') if t['default']=='yes' else dim('No ')
  fo=ok('Yes') if t['forced']=='yes' else dim('No ')
  en=ok('ON ') if t['enabled']=='yes' else er('OFF')
  nm=_pad(t['name'] if t['name'] else '-',18)
  print('  '+_pad(i,2)+'  '+_pad(t['codec'],20)+'  '+_pad(t['type'],8)+'  '+_pad(t['language'],4)+'  '+nm+'  '+_pad(t['track_id'],3)+'  '+de+'  '+fo+'  '+en)
 print()
def propedit(workfile,args):
 r=subprocess.run(['mkvpropedit',str(workfile)]+args,capture_output=True,text=True,timeout=120)
 out=(r.stdout+'\n'+r.stderr).strip()
 return (r.returncode==0,out[-400:] if out else '')
def track_sel(t):
 if t.get('uid'):return 'track:='+str(t['uid'])
 return 'track:'+str(t['track_id']+1)
def sel_file():
 ci();hdr('PILIH FILE (MKV)')
 fs=[]
 for d in [UPLOAD,OUTPUT,Path('/content/extracts')]:
  if not d.exists():continue
  for p in sorted(d.rglob('*')):
   if p.is_file() and p.suffix.lower() in MKVOK:fs.append(p)
 if not fs:print('\n  '+er('Tidak ada file MKV.'));input('  Enter...');return None
 print()
 for i,f in enumerate(fs):
  size=f.stat().st_size/1024/1024
  print('  ['+str(i)+'] '+f.name+'  '+dim(str(int(size))+'MB'))
 print()
 c=input('  Pilih: ').strip()
 try:
  idx=int(c)
  if 0<=idx<len(fs):return fs[idx]
 except:pass
 return None
def make_workfile(src):
 out=OUTPUT/(src.stem+'.meta.mkv')
 if out.exists():
  c=input('  File kerja sudah ada: '+out.name+' | [Y] pakai  [N] copy ulang: ').strip().upper()
  if c in ('','Y'):return out
 print('  Copy ke '+out.name+' ...')
 import shutil
 shutil.copy2(src,out)
 return out
def edit_track(t,workfile,all_tracks,changes):
 while True:
  ci()
  print('\n  EDIT TRACK ['+t['type']+' TID '+str(t['track_id'])+'] '+t['codec'])
  print('  File kerja: '+Path(workfile).name+'\n')
  print('    [1] Language : '+t['language']+' ('+L.get(t['language'],'?')+')')
  print('    [2] Nama     : '+(t['name'] or '(kosong)'))
  print('    [3] Default  : '+t['default'])
  print('    [4] Forced   : '+t['forced'])
  print('    [5] Enabled  : '+t['enabled'])
  print('    [6] Jadikan SATU-SATUNYA default tipe ini')
  print('\n    [0] Kembali\n')
  c=input('  Pilih: ').strip()
  if c=='0':return
  sel=track_sel(t)
  if c=='1':
   print('\n  Codes: '+', '.join(sorted(L.keys())))
   v=input('  Language ['+t['language']+']: ').strip()
   if v:
    okm,msg=propedit(workfile,['--edit',sel,'--set','language='+v])
    if okm:t['language']=v;changes.append('TID '+str(t['track_id'])+' lang='+v);print(ok('  OK'))
    else:print(er('  Gagal: '+msg))
    input('  Enter...')
  elif c=='2':
   v=input('  Nama (kosong=hapus) ['+t['name']+']: ')
   args=['--edit',sel,'--delete','name'] if not v.strip() else ['--edit',sel,'--set','name='+v]
   okm,msg=propedit(workfile,args)
   if okm:t['name']=v.strip();changes.append('TID '+str(t['track_id'])+' name='+v.strip());print(ok('  OK'))
   else:print(er('  Gagal: '+msg))
   input('  Enter...')
  elif c in ('3','4','5'):
   key={'3':'default','4':'forced','5':'enabled'}[c]
   prop={'3':'flag-default','4':'flag-forced','5':'flag-enabled'}[c]
   nv='no' if t[key]=='yes' else 'yes'
   okm,msg=propedit(workfile,['--edit',sel,'--set',prop+'='+'1' if nv=='yes' else '0'])
   if okm:t[key]=nv;changes.append('TID '+str(t['track_id'])+' '+key+'='+nv);print(ok('  OK'))
   else:print(er('  Gagal: '+msg))
   input('  Enter...')
  elif c=='6':
   bad=False
   for o in all_tracks:
    if o['type']==t['type'] and o is not t:
     okm,msg=propedit(workfile,['--edit',track_sel(o),'--set','flag-default=0'])
     if okm:o['default']='no';changes.append('TID '+str(o['track_id'])+' default=no')
     else:bad=True;print(er('  Gagal TID '+str(o['track_id'])+': '+msg))
   okm,msg=propedit(workfile,['--edit',sel,'--set','flag-default=1'])
   if okm:t['default']='yes';changes.append('TID '+str(t['track_id'])+' default=yes (sole)');print(ok('  OK'))
   else:bad=True;print(er('  Gagal: '+msg))
   if bad:input('  Enter...')
def menu_meta():
 src=sel_file()
 if not src:return
 workfile=make_workfile(src)
 if not workfile:return
 changes=[]
 while True:
  tracks,title,chapters=probe_meta(workfile)
  if not tracks:print(er('  Tidak ada track.'));input('  Enter...');return
  for i,t in enumerate(tracks):t['idx']=i
  ci();hdr('METADATA EDITOR')
  print('\n  File kerja: '+Path(workfile).name)
  print('  Judul file: '+(title or '-'))
  chinfo=str(len(chapters))+' chapter' if chapters else 'tanpa chapter'
  tginfo='ada tags' if has_tags(workfile) else 'tanpa tags'
  print('  '+chinfo+' | '+tginfo)
  show_tracks(tracks)
  print('  [0-9]  Edit track')
  print('  [T]    Judul file')
  print('  [C]    Rename chapter')
  print('  [G]    Hapus SEMUA tags')
  print('  [V]    Verify ulang')
  print('  [Q]    Selesai')
  print()
  c=input('  > ').strip().upper()
  if c=='Q':break
  elif c=='T':
   v=input('  Judul baru (kosong=hapus) ['+title+']: ')
   args=['--edit','info','--delete','title'] if not v.strip() else ['--edit','info','--set','title='+v]
   okm,msg=propedit(workfile,args)
   if okm:changes.append('title='+v.strip());print(ok('  OK'))
   else:print(er('  Gagal: '+msg))
   input('  Enter...')
  elif c=='C':
   if not chapters:print(er('  File ini tidak punya chapter.'));input('  Enter...');continue
   print()
   for ch in chapters:print('  ['+str(ch['no'])+'] '+ch['name'])
   print()
   v=input('  Nomor chapter: ').strip()
   try:n=int(v)
   except:continue
   if not (1<=n<=len(chapters)):continue
   nv=input('  Nama baru: ').strip()
   if not nv:continue
   okm,msg=propedit(workfile,['--edit','chapter:'+str(n),'--set','name='+nv])
   if okm:changes.append('chapter '+str(n)+'='+nv);print(ok('  OK'))
   else:print(er('  Gagal: '+msg))
   input('  Enter...')
  elif c=='G':
   go=input('  Hapus SEMUA tags? Ketik YA: ').strip()
   if go=='YA':
    okm,msg=propedit(workfile,['--tags','all:'])
    if okm:changes.append('tags cleared');print(ok('  OK'))
    else:print(er('  Gagal: '+msg))
    input('  Enter...')
  elif c=='V':continue
  elif c.isdigit():
   i=int(c)
   if 0<=i<len(tracks):edit_track(tracks[i],workfile,tracks,changes)
 if changes:
  mb=Path(workfile).stat().st_size/1024/1024
  print(ok('\n  Selesai: '+str(len(changes))+' perubahan -> '+Path(workfile).name+' ('+str(round(mb,1))+' MB)'))
  msg='<b>Metadata selesai</b>\n'+Path(workfile).name+'\n'+str(len(changes))+' perubahan'
  tg_send(msg)
 else:print('\n  Tidak ada perubahan.')
 input('\n  Enter...')
def has_tags(workfile):
 try:
  r=subprocess.run(['mkvmerge','-J',str(workfile)],capture_output=True,text=True,timeout=30)
  d=json.loads(r.stdout)
  return bool(d.get('tags'))
 except:return False
def get_gofile_token():
 return get_secret('GOFILE_API_TOKEN')
def gofile_wt(agent,token):
 import hashlib,time
 slot=int(time.time())//14400
 return hashlib.sha256((agent+'::en-US::'+token+'::'+str(slot)+'::12af056dacea0b').encode()).hexdigest()
def gofile_api_list(url,password,token):
 payload={'url':url,'password':password,'expiresInSeconds':3600,'filePage':0,'fileSize':100}
 headers={'Authorization':'Bearer '+token,'Content-Type':'application/json'}
 r=requests.post('https://go.filmbeehub.workers.dev/api/v1/generate',json=payload,headers=headers,timeout=60)
 res=r.json()
 if not res.get('ok'):print(er('  Gagal: '+str(res.get('error','unknown'))));return []
 data=res.get('data',{})
 if data.get('downloadLinks'):return data['downloadLinks']
 share_url=data.get('shareUrl','')
 if share_url:
  sid=share_url.rstrip('/').split('/')[-1]
  fd=requests.get('https://go.filmbeehub.workers.dev/api/data/'+sid,headers={'User-Agent':'Mozilla/5.0'},timeout=30).json()
  out=[]
  for g in fd.get('groups',[]):out.extend(g.get('files',[]))
  return out
 return []
def gofile_dl_one(link,dest_dir,tries=3):
 durl=link.get('downloadUrl','');name=link.get('name','file')
 if not durl:print('  Skip (no URL).');return None
 dest=dest_dir/name;part=dest_dir/(name+'.part')
 if dest.exists() and dest.stat().st_size>0:print('  SKIP '+name+' (sudah ada)');return dest
 for att in range(1,tries+1):
  try:
   print('  Downloading '+name+'...'+('' if att==1 else ' (coba '+str(att)+')'))
   rr=requests.get(durl,stream=True,timeout=600)
   rr.raise_for_status()
   total=0;fh=open(part,'wb')
   for ch in rr.iter_content(chunk_size=1024*1024):
    if ch:fh.write(ch);total+=len(ch)
   fh.close()
   if total==0:raise Exception('0 byte')
   os.rename(part,dest)
   print('  OK '+name+' ('+str(round(total/1024/1024,1))+' MB)')
   return dest
  except Exception as e:
   try:fh.close()
   except:pass
   try:
    if part.exists():os.remove(part)
   except:pass
   if att<tries:print('  Retry...');time.sleep(10*att)
   else:print(er('  Gagal: '+name))
 return None
def gofile_direct_fetch(url,password):
 import hashlib
 m=re.search(r'gofile\.io/d/(\w+)',url)
 if not m:return None,'Link tidak valid',None
 cid=m.group(1)
 pw=hashlib.sha256(password.encode()).hexdigest() if password else None
 agent='Mozilla/5.0'
 s=requests.Session()
 s.headers.update({'Accept-Encoding':'gzip','User-Agent':agent,'Connection':'keep-alive','Accept':'*/*','Origin':'https://gofile.io','Referer':'https://gofile.io/'})
 try:
  r=s.post('https://api.gofile.io/accounts',headers={'X-Website-Token':gofile_wt(agent,''),'X-BL':'en-US'},timeout=20)
  tok=r.json()['data']['token']
 except Exception as e:return None,'Guest gagal: '+str(e)[:120],None
 s.cookies.set('Cookie','accountToken='+tok)
 s.headers.update({'Authorization':'Bearer '+tok})
 files=[]
 try:
  def walk(x):
   u='https://api.gofile.io/contents/'+x+'?cache=true'
   if pw:u=u+'&password='+pw
   r=s.get(u,headers={'X-Website-Token':gofile_wt(agent,tok),'X-BL':'en-US'},timeout=30)
   d=r.json()
   if d.get('status')!='ok':raise Exception(str(d.get('status'))[:60])
   data=d['data']
   if data.get('passwordStatus','passwordOk')!='passwordOk' and 'password' in data:raise Exception('password salah')
   if data.get('type')!='folder':
    if data.get('link'):files.append({'name':data['name'],'size':data.get('size',0),'link':data['link']})
    return
   for ch in (data.get('children',{}) or {}).values():
    if ch.get('type')=='folder':walk(ch['id'])
    elif ch.get('link'):files.append({'name':ch['name'],'size':ch.get('size',0),'link':ch['link']})
  walk(cid)
 except Exception as e:return None,'List gagal: '+str(e)[:150],None
 return files,None,tok
def gofile_direct_dl(files,tok,dest_dir):
 ok_n=0
 hdr={'User-Agent':'Mozilla/5.0','Referer':'https://gofile.io/','Origin':'https://gofile.io','Cookie':'accountToken='+tok}
 for f in files:
  name=f['name'];dest=dest_dir/name;part=dest_dir/(name+'.part')
  if dest.exists() and dest.stat().st_size>0:print('  SKIP '+name);ok_n+=1;continue
  done=False
  for att in range(1,4):
   try:
    print('  Direct '+name+'...')
    rr=requests.get(f['link'],headers=hdr,stream=True,timeout=600)
    rr.raise_for_status()
    total=0;fh=open(part,'wb')
    for ch in rr.iter_content(chunk_size=1024*1024):
     if ch:fh.write(ch);total+=len(ch)
    fh.close()
    if total==0:raise Exception('0 byte')
    os.rename(part,dest)
    print('  '+ok('OK')+' '+name)
    done=True;break
   except Exception as e:
    try:fh.close()
    except:pass
    try:
     if part.exists():os.remove(part)
    except:pass
    if att<3:time.sleep(10*att)
  if done:ok_n+=1
  else:print(er('  Gagal: '+name))
 return ok_n

def dl_gofile():
 print('  Redirecting ke haru-download...');subprocess.run(['haru-download'])

def dl_gofile():
 print('  Redirecting ke haru-download...');subprocess.run(['haru-download'])
def dl_drive():dl_gofile()

def dl_gofile():
 print('  Redirecting ke haru-download...');subprocess.run(['haru-download'])
def dl_drive():dl_gofile()
def dl_url():dl_gofile()

def dl_gofile():
 print('  Redirecting ke haru-download...');subprocess.run(['haru-download'])
def dl_drive():dl_gofile()
def dl_url():dl_gofile()
def menu_download():dl_gofile()

def menu_download():dl_gofile()

def dl_url():dl_gofile()
def menu_download():dl_gofile()

def dl_drive():
 hdr('DOWNLOAD - Google Drive')
 url=input('\n  Link GDrive: ').strip()
 if not url:return
 print('  Downloading...')
 subprocess.run(['gdown','--folder','-O',str(UPLOAD),'--remaining-ok',url],timeout=600)
 print(ok('  Selesai!'));input('  Enter...')
def dl_url():
 hdr('DOWNLOAD - Direct URL')
 url=input('\n  Direct URL: ').strip()
 if not url:return
 subprocess.run(['wget','-q','-P',str(UPLOAD),'--content-disposition','--no-check-certificate',url],timeout=600)
 print(ok('  Selesai!'));input('  Enter...')
def menu_download():
 while True:
  ci();hdr('DOWNLOAD')
  print()
  print('  [1] Gofile')
  print('  [2] Google Drive')
  print('  [3] Direct URL')
  print()
  print('  [0] Kembali')
  print()
  c=input('  Pilih: ').strip()
  if c=='0':return
  elif c=='1':dl_gofile()
  elif c=='2':dl_drive()
  elif c=='3':dl_url()
def gdrive_secret(k):
 return get_secret(k)
def upload_gofile():
 print('  Redirecting ke haru-upload...');subprocess.run(['haru-upload'])

def upload_gofile():
 print('  Redirecting ke haru-upload...');subprocess.run(['haru-upload'])
def upload_drive():upload_gofile()

def upload_gofile():
 print('  Redirecting ke haru-upload...');subprocess.run(['haru-upload'])
def upload_drive():upload_gofile()
def menu_upload():upload_gofile()

def menu_upload():upload_gofile()

def gdrive_token(cid,sec,ref):
 try:
  r=requests.post('https://oauth2.googleapis.com/token',data={'client_id':cid,'client_secret':sec,'refresh_token':ref,'grant_type':'refresh_token'},timeout=15)
  return r.json().get('access_token')
 except:return None
def parse_drive_folder(tok,folder):
 import re
 m=re.search(r'/folders/([A-Za-z0-9_-]+)',folder)
 if m:return m.group(1)
 if len(folder)>20 and '/' not in folder and ' ' not in folder:return folder
 if tok:return gdrive_find_folder(tok,folder)
 return None
def gdrive_find_folder(tok,name):
 try:
  q="name='"+name+"' and mimeType='application/vnd.google-apps.folder' and trashed=false"
  r=requests.get('https://www.googleapis.com/drive/v3/files',headers={'Authorization':'Bearer '+tok},params={'q':q,'fields':'files(id,name)'},timeout=15)
  fs=r.json().get('files',[])
  if fs:return fs[0]['id']
  meta={'name':name,'mimeType':'application/vnd.google-apps.folder'}
  r2=requests.post('https://www.googleapis.com/drive/v3/files',headers={'Authorization':'Bearer '+tok,'Content-Type':'application/json'},data=json.dumps(meta),timeout=15)
  return r2.json().get('id')
 except:return None
def gdrive_upload_file(tok,fpath,parent):
 size=fpath.stat().st_size
 meta={'name':fpath.name,'parents':[parent]}
 try:
  r=requests.post('https://www.googleapis.com/upload/drive/v3/files?uploadType=resumable',headers={'Authorization':'Bearer '+tok,'Content-Type':'application/json','X-Upload-Content-Type':'application/octet-stream','X-Upload-Content-Length':str(size)},data=json.dumps(meta),timeout=30)
  uri=r.headers.get('Location')
  if not uri:return False
 except:return False
 CH=64*1024*1024 if size>100*1024*1024 else 16*1024*1024
 up=0;t0=time.time()
 try:
  fh=open(fpath,'rb')
  while up<size:
   ch=fh.read(CH)
   if not ch:break
   end=up+len(ch)-1
   rr=requests.put(uri,headers={'Content-Range':'bytes '+str(up)+'-'+str(end)+'/'+str(size),'Content-Length':str(len(ch))},data=ch,timeout=120)
   if rr.status_code in (200,201):up+=len(ch);break
   elif rr.status_code==308:
    up+=len(ch)
    el=time.time()-t0;sp=up/el/1024/1024 if el>0 else 0
    print('  '+str(round(up/size*100,1))+'%  '+str(round(sp,1))+' MB/s')
   else:fh.close();return False
  fh.close()
 except:return False
 print(ok('  100% Selesai.'))
 return True

def upload_drive():
 hdr('UPLOAD - Google Drive')
 all_files=[]
 for d in [UPLOAD,OUTPUT,Path('/content/extracts'),Path('/content/downloads')]:
  if d.exists():
   for f in sorted(d.rglob('*')):
    if f.is_file() and f.suffix.lower() in V|A|S:all_files.append((d,f))
 if not all_files:print(er('  Tidak ada file untuk di-upload.'));input('  Enter...');return
 print()
 idx=0
 for d in [UPLOAD,OUTPUT,Path('/content/extracts'),Path('/content/downloads')]:
  grp=[(dd,f) for dd,f in all_files if dd==d]
  if not grp:continue
  print('  ['+d.name+'/]  ('+str(len(grp))+' file)')
  for dd,f in grp:
   size=f.stat().st_size/1024/1024
   print('    ['+str(idx)+'] '+f.name+'  '+dim(str(int(size))+'MB'))
   idx+=1
  print()
 flat=[f for dd,f in all_files]
 c=input('  Pilih (* semua / 0,1,2 / 0-3 / Q batal): ').strip().upper()
 if c=='Q':return
 if c=='*':targets=flat
 else:
  try:
   nums=[]
   for part in c.split(','):
    part=part.strip()
    if '-' in part:a,b=part.split('-',1);nums.extend(range(int(a),int(b)+1))
    else:nums.append(int(part))
   targets=[flat[n] for n in nums if 0<=n<len(flat)]
  except:print('  Input tidak valid.');input('  Enter...');return
  if not targets:return
 cid=gdrive_secret('GDRIVE_CLIENT_ID');sec=gdrive_secret('GDRIVE_CLIENT_SECRET');ref=gdrive_secret('GDRIVE_REFRESH_TOKEN')
 parent_id=gdrive_secret('GDRIVE_FOLDER_ID') or '1pjpd63PTFvwYd8iI7dvMwcU-e_LMqvUE'
 if not(cid and sec and ref):
  print(er('  Secret GDrive tidak kebaca.'));print('  Aktifkan toggle secret + re-run cell Install.');input('  Enter...');return
 print('  Auth via API...')
 tok=gdrive_token(cid,sec,ref)
 if not tok:print(er('  Gagal dapat access token.'));return
 import re
 m=re.search(r'/folders/([A-Za-z0-9_-]+)',parent_id)
 if m:parent_id=m.group(1)
 elif len(parent_id)<20:
  q="name='"+parent_id+"' and mimeType='application/vnd.google-apps.folder' and trashed=false"
  try:
   r=requests.get('https://www.googleapis.com/drive/v3/files',headers={'Authorization':'Bearer '+tok},params={'q':q,'fields':'files(id)'},timeout=15)
   fs=r.json().get('files',[])
   if fs:parent_id=fs[0]['id']
  except:pass
 sub=input('  Subfolder ['+dim('langsung ke parent')+']: ').strip()
 target=parent_id
 if sub:
  try:
   q2="name='"+sub+"' and '"+parent_id+"' in parents and mimeType='application/vnd.google-apps.folder' and trashed=false"
   r2=requests.get('https://www.googleapis.com/drive/v3/files',headers={'Authorization':'Bearer '+tok},params={'q':q2,'fields':'files(id)'},timeout=15)
   fs2=r2.json().get('files',[])
   if fs2:target=fs2[0]['id']
   else:
    meta={'name':sub,'mimeType':'application/vnd.google-apps.folder','parents':[parent_id]}
    r3=requests.post('https://www.googleapis.com/drive/v3/files',headers={'Authorization':'Bearer '+tok,'Content-Type':'application/json'},data=json.dumps(meta),timeout=15)
    nid=r3.json().get('id')
    if nid:target=nid;print('  Subfolder dibuat: '+sub)
    else:print(er('  Gagal buat subfolder.'))
  except:print(er('  Error buat subfolder.'))
 ok_n=0;fail=[]
 for f in targets:
  print('  Upload '+f.name+' ('+str(round(f.stat().st_size/1024/1024,1))+'MB)...')
  if gdrive_upload_file(tok,f,target):ok_n+=1;print('  '+ok('ok')+' '+f.name)
  else:fail.append(f.name);print('  '+er('gagal')+' '+f.name)
 if ok_n:tg_send('<b>Upload GDrive</b>\n'+str(ok_n)+' file berhasil')
 if fail:print(er('  Gagal: '+', '.join(fail)))
 input('\n  Enter...')



def menu_upload():
 ci();hdr('UPLOAD')
 print()
 print('  [1] Gofile  (folder gabungan)')
 print('  [2] Google Drive (multi-file + subfolder)')
 print()
 print('  [0] Kembali')
 print()
 c=input('  Pilih: ').strip()
 if c=='0':return
 elif c=='1':upload_gofile()
 elif c=='2':upload_drive()


def menu_browse():
 ci();hdr('BROWSE FILES')
 print()
 for d in [UPLOAD,OUTPUT]:
  print('  ['+str(d)+']')
  subprocess.run(['ls','-lh',str(d)])
  print()
 input('  Enter...')
def menu_list():
 src=sel_file()
 if not src:return
 tracks,title,chapters=probe_meta(src)
 if not tracks:print(er('  Tidak ada track.'));input('  Enter...');return
 ci();hdr('LIST TRACKS - '+src.name)
 print('  Judul: '+(title or '-'))
 if chapters:
  print('  Chapters: '+str(len(chapters)))
  for ch in chapters:print('    '+str(ch['no'])+'. '+ch['name'])
 show_tracks([dict(t,**{'idx':i}) for i,t in enumerate(tracks)])
 input('  Enter...')
def page_out(text):
 ls=text.splitlines()
 if len(ls)>80:
  import tempfile
  tp=os.path.join(tempfile.gettempdir(),'mi.txt')
  open(tp,'w',encoding='utf-8',errors='ignore').write(text)
  print('  Output panjang ('+str(len(ls))+' baris) -> less, q keluar, panah scroll')
  try:subprocess.run(['less','-M','-X',tp])
  except:print(text)
 else:print(text)

def telegraph_upload(title,text):
 try:
  r=requests.post('https://api.telegra.ph/createAccount',data={'short_name':'haru','author_name':'haru-meta'},timeout=20)
  tok=r.json()['result']['access_token']
 except Exception as e:print(er('  Telegraph gagal.'));return None
 try:
  nodes=json.dumps([{'tag':'pre','children':[text[:60000]]}])
  r=requests.post('https://api.telegra.ph/createPage',data={'access_token':tok,'title':title[:60],'author_name':'haru-meta','content':nodes},timeout=30)
  d=r.json()
  if d.get('ok'):print(ok('  '+d['result']['url']));return d['result']['url']
 except Exception as e:print(er('  Telegraph error.'))
 return None
def telegraph_bulk(title,sections):
 pages=[];cur=[];curlen=0
 for name,text in sections:
  bl=len(name)+len(text)+100
  if cur and curlen+bl>58000:
   pages.append(cur);cur=[];curlen=0
  cur.append((name,text));curlen+=bl
 if cur:pages.append(cur)
 urls=[]
 for i,pg in enumerate(pages):
  nodes=[]
  for name,text in pg:
   nodes.append({'tag':'h4','children':[name]})
   nodes.append({'tag':'pre','children':[text[:60000]]})
  try:
   r=requests.post('https://api.telegra.ph/createAccount',data={'short_name':'haru','author_name':'haru-meta'},timeout=20)
   tok=r.json()['result']['access_token']
   t=title+(' (%d/%d)'%(i+1,len(pages)) if len(pages)>1 else '')
   r=requests.post('https://api.telegra.ph/createPage',data={'access_token':tok,'title':t[:60],'author_name':'haru-meta','content':json.dumps(nodes)},timeout=30)
   d=r.json()
   if d.get('ok'):urls.append(d['result']['url']);print(ok('  Hal '+str(i+1)+': '+d['result']['url']))
  except Exception as e:print(er('  Gagal hal '+str(i+1)))
 return urls
def mi_files():
 fs=[]
 for d in [UPLOAD,OUTPUT]:
  if d.exists():
   for p in sorted(d.rglob('*')):
    if p.is_file() and p.suffix.lower() in V|A|S|MKVOK:fs.append((d,p))
 return fs
def menu_info():
 ci();hdr('MEDIAINFO')
 print()
 print('  [1] Pilih file (satuan/*)')
 print('  [2] Bulk 1 folder -> telegra.ph gabungan')
 print()
 print('  [0] Kembali')
 print()
 c=input('  Pilih: ').strip()
 if c=='0':return
 if c=='2':return mi_bulk()
 items=mi_files()
 if not items:print(er('  Tidak ada file.'));input('  Enter...');return
 print()
 idx=0
 for d in [UPLOAD,OUTPUT]:
  grp=[f for dd,f in items if dd==d]
  if not grp:continue
  print('  ['+d.name+'/]')
  for f in grp:
   size=f.stat().st_size/1024/1024
   print('  ['+str(idx)+'] '+f.name+'  '+dim(str(int(size))+'MB'))
   idx+=1
  print()
 flat=[f for dd,f in items]
 c=input('  Pilih file (atau * semua): ').strip()
 if c=='*':targets=flat
 else:
  try:
   idx=int(c)
   if 0<=idx<len(flat):targets=[flat[idx]]
   else:return
  except:return
 fmt=input('  Format (T=text, J=json) [T]: ').strip().upper() or 'T'
 saved=[]
 for f in targets:
  cmd=['mediainfo']
  if fmt=='J':cmd.append('--Output=JSON')
  cmd.append(str(f))
  r=subprocess.run(cmd,capture_output=True,text=True,timeout=30)
  page_out(r.stdout)
  saved.append((f.name,r.stdout))
 if saved:
  u=input('\n  Upload ke telegra.ph? [Y/n]: ').strip().lower()
  if u in ('','y'):
   links=[]
   for name,text in saved:
    url=telegraph_upload('MediaInfo - '+name,text)
    if url:links.append((name,url))
   if links:
    msg='<b>MediaInfo</b>'
    for name,url in links:msg=msg+'\n'+name+'\n'+url
    tg_send(msg)
 input('  Enter...')
def mi_bulk():
 ci();hdr('BULK MEDIAINFO')
 dirs=[d for d in [UPLOAD,OUTPUT] if d.exists()]
 if not dirs:return
 print()
 for i,d in enumerate(dirs):print('  ['+str(i)+'] '+str(d))
 print()
 c=input('  Folder: ').strip()
 try:d=dirs[int(c)]
 except:return
 fs=[p for p in sorted(d.rglob('*')) if p.is_file() and p.suffix.lower() in V|A|S|MKVOK]
 if not fs:print(er('  Kosong.'));input('  Enter...');return
 print('\n  Proses '+str(len(fs))+' file...')
 sections=[]
 for f in fs:
  r=subprocess.run(['mediainfo',str(f)],capture_output=True,text=True,timeout=30)
  sections.append((f.name,r.stdout))
  print('  ok '+f.name)
 print()
 urls=telegraph_bulk('MediaInfo - '+d.name+' ('+str(len(fs))+' file)',sections)
 if urls:
  msg='<b>Bulk MediaInfo</b>\n'+str(len(fs))+' file'
  for u in urls:msg=msg+'\n'+u
  tg_send(msg)
 input('\n  Enter...')
def main():
 load_secrets()
 while True:
  ci()
  print('\n'+'\033[96m'+'='*62+'\033[0m')
  print('\033[96m  haru-metadata v2026.09.08b -- Edit Metadata MKV\033[0m')
  print('\033[96m'+'='*62+'\033[0m')
  print()
  print('  [1]  Download       -- Gofile / GDrive / URL')
  print('  [2]  Metadata       -- Pilih file, edit, instant')
  print('  [3]  List Tracks    -- Lihat track + chapter + judul')
  print('  [4]  MediaInfo      -- Satuan / bulk folder -> telegra.ph')
  print('  [5]  Upload         -- Upload hasil edit')
  print('  [6]  Browse         -- Lihat isi folder')
  print()
  print('  [Q]  Keluar')
  secs=[]
  if get_secret('GOFILE_API_TOKEN'):secs.append('gofile')
  if get_secret('GDRIVE_REFRESH_TOKEN'):secs.append('gdrive')
  if get_secret('OWNER_ID') and get_secret('HARU_BOT_TOKEN'):secs.append('telegram')
  print()
  print('  Secrets: '+(dim(', '.join(secs)) if secs else er('KOSONG! re-run cell 1B')))
  print()
  c=input('  Pilih: ').strip().upper()
  if c=='Q':print('\n  Bye!');sys.exit(0)
  elif c=='1':menu_download()
  elif c=='2':menu_meta()
  elif c=='3':menu_list()
  elif c=='4':menu_info()
  elif c=='5':menu_upload()
  elif c=='6':menu_browse()
if __name__=='__main__':main()
'''


HARU_AUTORENAME_SCRIPT = r'''#!/usr/bin/env python3
import subprocess,sys,os,re,glob,json,time
from pathlib import Path
V={'.mkv','.mp4','.avi','.mov','.webm','.flv','.wmv','.ts','.m4v'}
A={'.mp3','.aac','.flac','.wav','.ogg','.opus','.mka','.ac3','.dts','.eac3','.m4a'}
S={'.srt','.ass','.ssa','.sub','.idx','.sup','.vtt','.pgs','.scc','.sami'}
ALL_EXT=V|A|S
UPLOAD=Path('/content/uploads')
OUTPUT=Path('/content/output')
EXTRACTS=Path('/content/extracts')
def ci():os.system('cls' if os.name=='nt' else 'clear')
def ok(t):return '\033[92m'+t+'\033[0m'
def er(t):return '\033[91m'+t+'\033[0m'
def dim(t):return '\033[90m'+t+'\033[0m'
def hdr(title):print('\n'+'='*62);print('  '+title);print('='*62)

def clean_filename(name):
 name=name.strip()
 name=re.sub(r'\[([A-Za-z0-9]+)\]',r'[\1] ',name)
 name=re.sub(r'\s+',' ',name)
 name=re.sub(r'\(Dual Audio\)','(Dual-Audio)',name)
 name=re.sub(r'\(Dual Audio ','(Dual-Audio ',name)
 name=re.sub(r' (Dual Audio) ',' (Dual-Audio) ',name)
 m=re.search(r'(?<!\d)(\d{1,3})(?!\d)',name)
 if m:
  ep=m.group(1).zfill(2)
  before=name[:m.start()]
  after=name[m.end():]
  if not re.search(r'[Ss]\d+[Ee]\d+',name):
   season='01'
   sm=re.search(r'[Ss](\d{1,2})',before)
   if sm:season=sm.group(1).zfill(2)
   name=before+'S'+season+'E'+ep+after
 name=re.sub(r'\s*\(\s*',' (',name)
 name=re.sub(r'\s*\)\s*',') ',name)
 name=re.sub(r'  +',' ',name)
 name=name.strip()
 return name

def pick_folder():
 ci();hdr('AUTO RENAME - Pilih Folder')
 print()
 print('  [1] /content/uploads')
 print('  [2] /content/output')
 print('  [3] /content/extracts')
 print('  [4] Semua folder')
 print('  [Q] Kembali')
 print()
 c=input('  Pilih: ').strip().upper()
 if c=='Q':return None
 if c=='1':return UPLOAD
 if c=='2':return OUTPUT
 if c=='3':return EXTRACTS
 if c=='4':return [UPLOAD,OUTPUT,EXTRACTS]
 return None

def scan_files(folders):
 if not isinstance(folders,list):folders=[folders]
 files=[]
 for d in folders:
  if not d.exists():continue
  for p in sorted(d.rglob('*')):
   if not p.is_file():continue
   if p.suffix.lower() in ALL_EXT:
    cleaned=clean_filename(p.name)
    if cleaned!=p.name:files.append((p,cleaned))
 return files

def show_files(files):
 print()
 print('  No  Original                                          -> Cleaned')
 print('  '+'-'*80)
 for i,(orig,cleaned) in enumerate(files):
  print('  '+str(i).ljust(4)+orig.name[:50].ljust(52)+'-> '+cleaned[:40])

def do_rename(files,sel=None):
 ok_n=0
 targets=files if sel is None else [(files[i]) for i in sel if 0<=i<len(files)]
 for orig,cleaned in targets:
  new_path=orig.parent/cleaned
  if new_path.exists() and new_path!=orig:
   print('  Skip (exists): '+cleaned);continue
  try:
   orig.rename(new_path)
   print('  '+ok('OK')+' '+orig.name+' -> '+cleaned)
   ok_n+=1
  except Exception as e:print('  '+er('ERR')+' '+str(e)[:60])
 print('\n  Renamed: '+str(ok_n)+'/'+str(len(targets)))

def main():
 while True:
  folders=pick_folder()
  if folders is None:return
  files=scan_files(folders)
  if not files:
   print('  Tidak ada file yang perlu di-rename.');input('  Enter...');continue
  while True:
   ci();hdr('AUTO RENAME')
   show_files(files)
   print()
   print('  [Y] Rename semua   [nomor] pilih (0,2,5)   [P] Preview   [F] Ganti folder   [Q] Kembali')
   print()
   c=input('  > ').strip().upper()
   if c=='Q':break
   if c=='F':break
   if c=='P':
    for orig,cleaned in files:
     print('  '+orig.name)
     print('    -> '+cleaned)
     print()
    input('  Enter...');continue
   if c=='Y':
    do_rename(files)
    input('  Enter...');continue
   try:
    sel=set()
    for part in c.split(','):
     part=part.strip()
     if part.isdigit():sel.add(int(part))
    do_rename(files,sel)
    input('  Enter...')
   except:pass

if __name__=='__main__':main()
'''
HARU_DOWNLOAD_SCRIPT = r'''#!/usr/bin/env python3
import subprocess,sys,os,re,glob,json,time
import requests
from pathlib import Path
V={'.mkv','.mp4','.avi','.mov','.webm','.flv','.wmv','.ts','.m4v'}
A={'.mp3','.aac','.flac','.wav','.ogg','.opus','.mka','.ac3','.dts','.eac3','.m4a'}
S={'.srt','.ass','.ssa','.sub','.idx','.sup','.vtt','.pgs','.scc','.sami'}
L={'id':'Indonesian','en':'English','ja':'Japanese','ko':'Korean','zh':'Chinese','ms':'Malay','ar':'Arabic','de':'German','fr':'French','es':'Spanish','pt':'Portuguese','ru':'Russian','it':'Italian','th':'Thai','vi':'Vietnamese','hi':'Hindi','und':'Undetermined'}
UPLOAD=Path('/content/uploads')
OUTPUT=Path('/content/output')
UPLOAD.mkdir(exist_ok=True)
OUTPUT.mkdir(exist_ok=True)
def ci():os.system('cls' if os.name=='nt' else 'clear')
def ok(t):return '\033[92m'+t+'\033[0m'
def er(t):return '\033[91m'+t+'\033[0m'
def dim(t):return '\033[90m'+t+'\033[0m'
def hdr(title):print('\n'+'='*62);print('  '+title);print('='*62)
def load_secrets():
 try:
  if os.path.exists('/content/.haru_secrets.json'):
   d=json.load(open('/content/.haru_secrets.json'))
   for k,v in d.items():
    if v and not os.environ.get(k):os.environ[k]=str(v)
 except:pass
def get_secret(k):
 v=os.environ.get(k,'')
 if v:return v.strip()
 try:
  from google.colab import userdata
  t=userdata.get(k)
  if t:return str(t).strip()
 except:pass
 return ''
def get_gofile_token():
 return get_secret('GOFILE_API_TOKEN')
def gofile_api_generate(url,password,token):
 payload={'url':url,'password':password,'expiresInSeconds':3600,'filePage':0,'fileSize':100}
 headers={'Authorization':'Bearer '+token,'Content-Type':'application/json'}
 r=requests.post('https://go.filmbeehub.workers.dev/api/v1/generate',json=payload,headers=headers,timeout=60)
 return r.json()
def gofile_api_list(url,password,token):
 res=gofile_api_generate(url,password,token)
 if not res.get('ok'):
  print('  Gagal generate: '+str(res.get('error','unknown')))
  return []
 data=res.get('data',{})
 if data.get('downloadLinks'):return data['downloadLinks']
 share_url=data.get('shareUrl','')
 if share_url:
  sid=share_url.rstrip('/').split('/')[-1]
  print('  Share ID: '+sid)
  rr=requests.get('https://go.filmbeehub.workers.dev/api/data/'+sid,headers={'User-Agent':'Mozilla/5.0'},timeout=30)
  fd=rr.json()
  out=[]
  for g in fd.get('groups',[]):out.extend(g.get('files',[]))
  return out
 return []
def gofile_dl_one(link,tries=3):
 durl=link.get('downloadUrl','')
 name=link.get('name','file')
 if not durl:print('  Tidak ada download URL, skip.');return None
 dest=UPLOAD/name
 part=UPLOAD/(name+'.part')
 if dest.exists() and dest.stat().st_size>0:
  print('  SKIP '+name+' (sudah ada)')
  return dest
 for att in range(1,tries+1):
  try:
   print('  Downloading '+name+'...'+('' if att==1 else ' (coba '+str(att)+')'))
   rr=requests.get(durl,stream=True,timeout=600)
   rr.raise_for_status()
   total=0
   fh=open(part,'wb')
   for ch in rr.iter_content(chunk_size=1024*1024):
    if ch:fh.write(ch);total+=len(ch)
   fh.close()
   if total==0:raise Exception('0 byte')
   os.rename(part,dest)
   print('  OK '+name+' ('+str(total)+' bytes / '+str(round(total/1024/1024,1))+' MB)')
   return dest
  except Exception as e:
   try:fh.close()
   except:pass
   try:
    if part.exists():os.remove(part)
   except:pass
   if att<tries:
    wait=10*att
    print('  Gagal, retry '+str(wait)+' detik... ('+str(e)[:120]+')')
    time.sleep(wait)
   else:print(er('  Gagal: '+name+' - '+str(e)[:150]))
 return None
def gofile_wt(agent,token):
 import hashlib,time
 slot=int(time.time())//14400
 return hashlib.sha256((agent+'::en-US::'+token+'::'+str(slot)+'::12af056dacea0b').encode()).hexdigest()
def gofile_direct_fetch(url,password):
 import hashlib
 m=re.search(r'gofile\.io/d/(\w+)',url)
 if not m:return None,'Link tidak valid',None
 cid=m.group(1)
 pw=hashlib.sha256(password.encode()).hexdigest() if password else None
 agent='Mozilla/5.0'
 s=requests.Session()
 s.headers.update({'Accept-Encoding':'gzip','User-Agent':agent,'Connection':'keep-alive','Accept':'*/*','Origin':'https://gofile.io','Referer':'https://gofile.io/'})
 try:
  r=s.post('https://api.gofile.io/accounts',headers={'X-Website-Token':gofile_wt(agent,''),'X-BL':'en-US'},timeout=20)
  tok=r.json()['data']['token']
 except Exception as e:return None,'Guest account gagal: '+str(e)[:120],None
 s.cookies.set('Cookie','accountToken='+tok)
 s.headers.update({'Authorization':'Bearer '+tok})
 files=[]
 try:
  def walk(x):
   u='https://api.gofile.io/contents/'+x+'?cache=true'
   if pw:u=u+'&password='+pw
   r=s.get(u,headers={'X-Website-Token':gofile_wt(agent,tok),'X-BL':'en-US'},timeout=30)
   d=r.json()
   if d.get('status')!='ok':raise Exception(str(d.get('status'))[:60])
   data=d['data']
   if data.get('passwordStatus','passwordOk')!='passwordOk' and 'password' in data:raise Exception('password salah')
   if data.get('type')!='folder':
    if data.get('link'):files.append({'name':data['name'],'size':data.get('size',0),'link':data['link']})
    return
   for ch in (data.get('children',{}) or {}).values():
    if ch.get('type')=='folder':walk(ch['id'])
    elif ch.get('link'):files.append({'name':ch['name'],'size':ch.get('size',0),'link':ch['link']})
  walk(cid)
 except Exception as e:return None,'List gagal: '+str(e)[:150],None
 return files,None,tok
def gofile_direct_one(f,tok,dest_dir):
 name=f['name'];dest=dest_dir/name;part=dest_dir/(name+'.part')
 if dest.exists() and dest.stat().st_size>0:
  print('  SKIP '+name+' (sudah ada)');return True
 hdr={'User-Agent':'Mozilla/5.0','Referer':'https://gofile.io/','Origin':'https://gofile.io','Cookie':'accountToken='+tok}
 for att in range(1,4):
  try:
   print('  Direct '+name+'...'+('' if att==1 else ' (coba '+str(att)+')'))
   rr=requests.get(f['link'],headers=hdr,stream=True,timeout=600)
   rr.raise_for_status()
   total=0
   fh=open(part,'wb')
   for ch in rr.iter_content(chunk_size=1024*1024):
    if ch:fh.write(ch);total+=len(ch)
   fh.close()
   if total==0:raise Exception('0 byte')
   os.rename(part,dest)
   print('  OK '+name+' ('+str(total)+' bytes / '+str(round(total/1024/1024,1))+' MB)')
   return True
  except Exception as e:
   try:fh.close()
   except:pass
   try:
    if part.exists():os.remove(part)
   except:pass
   if att<3:
    print('  Gagal, retry... ('+str(e)[:120]+')')
    time.sleep(10*att)
   else:print(er('  Gagal: '+name+' - '+str(e)[:150]))
 return False
def gofile_direct_retry(url,pwd,names,dest_dir):
 print('  Coba jalur direct API untuk '+str(len(names))+' file...')
 files,err,tok=gofile_direct_fetch(url,pwd)
 if err:print(er('  Direct: '+err));return names
 targets=[f for f in files if f['name'] in names]
 if not targets:print(er('  Direct: file tidak ketemu di listing.'));return names
 still=[]
 for f in targets:
  if not gofile_direct_one(f,tok,dest_dir):still.append(f['name'])
 return still
def dl_gofile():
 hdr('DOWNLOAD - Gofile')
 url=input('\n  Link Gofile: ').strip()
 if not url:return
 pwd=input('  Password (kosong = tidak ada): ').strip()
 token=get_gofile_token()
 if not token:
  print(er('  GOFILE_API_TOKEN tidak ditemukan.'))
  print('  Aktifkan secret di menu Rahasia, jalankan ulang cell Install, lalu coba lagi.')
  input('  Enter...');return
 files=gofile_api_list(url,pwd,token)
 if not files:print('  Folder kosong / tidak bisa diakses.');return
 print('  Ditemukan '+str(len(files))+' file:')
 for i,ff in enumerate(files):print('    ['+str(i)+'] '+str(ff.get('name','?'))+' ('+str(ff.get('size','?'))+')')
 print()
 c=input('  Pilih (0 / 0,1 / 0-2 / * semua): ').strip()
 if c=='*':targets=files
 else:
  try:
   nums=[]
   for part in c.split(','):
    part=part.strip()
    if '-' in part:
     a,b=part.split('-',1);nums.extend(range(int(a),int(b)+1))
    else:nums.append(int(part))
   targets=[files[n] for n in nums if 0<=n<len(files)]
  except:print('  Input tidak valid.');return
  if not targets:print('  Tidak ada yang dipilih.');return
 fails=[]
 for link in targets:
  if not gofile_dl_one(link):fails.append(link.get('name','?'))
 if fails:
  print(er('  Gagal '+str(len(fails))+' file via proxy, coba jalur direct API...'))
  still=gofile_direct_retry(url,pwd,fails,UPLOAD)
  if still:
   print(er('  Tetap gagal '+str(len(still))+' file:'))
   for n in still:print('    - '+n)
  else:print(ok('  Semua download selesai (via direct)!'))
 else:print(ok('  Semua download selesai!'))
def dl_drive():
 hdr('DOWNLOAD - Google Drive')
 url=input('\n  Link GDrive: ').strip()
 if not url:return
 print('  Downloading...')
 r=subprocess.run(['gdown','--folder','-O',str(UPLOAD),'--remaining-ok',url],timeout=300)
 if r.returncode==0:print(ok('  Download selesai!'))
 else:print(er('  Download gagal (code '+str(r.returncode)+')'))
def dl_url():
 hdr('DOWNLOAD - Direct URL')
 url=input('\n  Direct URL: ').strip()
 if not url:return
 fname=input('  Filename (kosong = auto): ').strip() or None
 cmd=['wget','-q','-P',str(UPLOAD),'--content-disposition','--no-check-certificate']
 if fname:cmd.extend(['-O',str(UPLOAD/fname)])
 cmd.append(url)
 r=subprocess.run(cmd,timeout=300)
 if r.returncode==0:print(ok('  Download selesai!'))
 else:print(er('  Download gagal (code '+str(r.returncode)+')'))
def menu_download():
 while True:
  ci();hdr('DOWNLOAD')
  print()
  print('  [1] Gofile')
  print('  [2] Google Drive')
  print('  [3] Direct URL')
  print()
  print('  [0] Kembali')
  print()
  c=input('  Pilih: ').strip()
  if c=='0':return
  elif c=='1':dl_gofile()
  elif c=='2':dl_drive()
  elif c=='3':dl_url()
  input('\n  Enter...')
def main():
 load_secrets()
 menu_download()
if __name__=='__main__':main()
'''

HARU_UPLOAD_SCRIPT = r'''#!/usr/bin/env python3
import subprocess,sys,os,re,glob,json,time
import requests
from pathlib import Path
V={'.mkv','.mp4','.avi','.mov','.webm','.flv','.wmv','.ts','.m4v'}
A={'.mp3','.aac','.flac','.wav','.ogg','.opus','.mka','.ac3','.dts','.eac3','.m4a'}
S={'.srt','.ass','.ssa','.sub','.idx','.sup','.vtt','.pgs','.scc','.sami'}
L={'id':'Indonesian','en':'English','ja':'Japanese','ko':'Korean','zh':'Chinese','ms':'Malay','ar':'Arabic','de':'German','fr':'French','es':'Spanish','pt':'Portuguese','ru':'Russian','it':'Italian','th':'Thai','vi':'Vietnamese','hi':'Hindi','und':'Undetermined'}
UPLOAD=Path('/content/uploads')
OUTPUT=Path('/content/output')
UPLOAD.mkdir(exist_ok=True)
OUTPUT.mkdir(exist_ok=True)
def ci():os.system('cls' if os.name=='nt' else 'clear')
def ok(t):return '\033[92m'+t+'\033[0m'
def er(t):return '\033[91m'+t+'\033[0m'
def dim(t):return '\033[90m'+t+'\033[0m'
def hdr(title):print('\n'+'='*62);print('  '+title);print('='*62)
def load_secrets():
 try:
  if os.path.exists('/content/.haru_secrets.json'):
   d=json.load(open('/content/.haru_secrets.json'))
   for k,v in d.items():
    if v and not os.environ.get(k):os.environ[k]=str(v)
 except:pass
def get_secret(k):
 v=os.environ.get(k,'')
 if v:return v.strip()
 try:
  from google.colab import userdata
  t=userdata.get(k)
  if t:return str(t).strip()
 except:pass
 return ''
def get_gofile_token():
 return get_secret('GOFILE_API_TOKEN')
def gofile_upload_files(targets, folder_name=None):
 if not targets:return False,[]
 token=get_gofile_token()
 if not token:return False,[]
 try:
  r=requests.get('https://api.gofile.io/accounts',timeout=15)
  d=r.json()
  if d.get('status')!='ok':return False,[]
  account_token=d['data']['token']
 except:return False,[]
 folder_id=None
 if folder_name and len(targets)>1:
  try:
   r=requests.post('https://api.gofile.io/contents',headers={'Authorization':'Bearer '+account_token,'Content-Type':'application/json'},json={'type':'folder','title':folder_name},timeout=15)
   d=r.json()
   if d.get('status')=='ok':folder_id=d['data']['id']
  except:pass
 srv='store1'
 try:
  sv=requests.get('https://api.gofile.io/servers',headers={'Authorization':'Bearer '+account_token},timeout=15).json()
  if sv.get('status')=='ok':srv=sv['data']['servers'][0]['name']
 except:pass
 links=[]
 for f in targets:
  print('  Upload '+f.name+' ('+str(round(f.stat().st_size/1024/1024,1))+'MB) via '+srv+'...')
  cmd=['curl','-s','-F','file=@'+str(f)]
  if folder_id:cmd.extend(['-F','folderId='+folder_id])
  cmd.append('https://'+srv+'.gofile.io/uploadFile')
  r=subprocess.run(cmd,capture_output=True,text=True,timeout=600)
  try:
   data=json.loads(r.stdout)
   if data.get('status')=='ok':
    links.append((f.name,data['data']['downloadPage']))
    print('  '+ok('ok')+' '+f.name)
   else:print('  '+er('gagal')+' '+str(data)[:100])
  except:print('  '+er('gagal')+' '+f.name+' (no response)')
 if not links:return False,[]
 if len(links)==1:return True,[links[0]]
 return True,links
def gdrive_secret(k):
 return get_secret(k)
def gdrive_token(cid,sec,ref):
 try:
  r=requests.post('https://oauth2.googleapis.com/token',data={'client_id':cid,'client_secret':sec,'refresh_token':ref,'grant_type':'refresh_token'},timeout=15)
  return r.json().get('access_token')
 except:return None
def gdrive_upload_file(tok,fpath,parent):
 size=fpath.stat().st_size
 meta={'name':fpath.name,'parents':[parent]}
 try:
  r=requests.post('https://www.googleapis.com/upload/drive/v3/files?uploadType=resumable',headers={'Authorization':'Bearer '+tok,'Content-Type':'application/json','X-Upload-Content-Type':'application/octet-stream','X-Upload-Content-Length':str(size)},data=json.dumps(meta),timeout=30)
  uri=r.headers.get('Location')
  if not uri:print('  Gagal mulai sesi upload.');return False
 except Exception as e:print('  Error inisiasi: '+str(e)[:150]);return False
 CH=64*1024*1024 if size>100*1024*1024 else 16*1024*1024
 up=0;t0=time.time()
 try:
  fh=open(fpath,'rb')
  while up<size:
   ch=fh.read(CH)
   if not ch:break
   end=up+len(ch)-1
   rr=requests.put(uri,headers={'Content-Range':'bytes '+str(up)+'-'+str(end)+'/'+str(size),'Content-Length':str(len(ch))},data=ch,timeout=120)
   if rr.status_code in (200,201):up+=len(ch);break
   elif rr.status_code==308:
    up+=len(ch)
    el=time.time()-t0;sp=up/el/1024/1024 if el>0 else 0
    print('  '+str(round(up/size*100,1))+'%  '+str(round(sp,1))+' MB/s')
   else:print('  Upload error HTTP '+str(rr.status_code));fh.close();return False
  fh.close()
 except Exception as e:print('  Error upload: '+str(e)[:150]);return False
 print(ok('  100% Selesai.'))
 return True
def upload_gofile():
 hdr('UPLOAD - Gofile')
 all_files=[]
 for d in [UPLOAD,OUTPUT,Path('/content/extracts'),Path('/content/downloads')]:
  if d.exists():
   for f in sorted(d.rglob('*')):
    if f.is_file() and f.suffix.lower() in V|A|S:all_files.append((d,f))
 if not all_files:print(er('  Tidak ada file untuk di-upload.'));input('  Enter...');return
 print()
 idx=0
 for d in [UPLOAD,OUTPUT,Path('/content/extracts'),Path('/content/downloads')]:
  grp=[(dd,f) for dd,f in all_files if dd==d]
  if not grp:continue
  print('  ['+d.name+'/]  ('+str(len(grp))+' file)')
  for dd,f in grp:
   size=f.stat().st_size/1024/1024
   print('    ['+str(idx)+'] '+f.name+'  '+dim(str(int(size))+'MB'))
   idx+=1
  print()
 flat=[f for dd,f in all_files]
 c=input('  Pilih (* semua / 0,1,2 / 0-3 / Q batal): ').strip().upper()
 if c=='Q':return
 if c=='*':targets=flat
 else:
  try:
   nums=[]
   for part in c.split(','):
    part=part.strip()
    if '-' in part:a,b=part.split('-',1);nums.extend(range(int(a),int(b)+1))
    else:nums.append(int(part))
   targets=[flat[n] for n in nums if 0<=n<len(flat)]
  except:print('  Input tidak valid.');input('  Enter...');return
  if not targets:return
 if len(targets)>1:
  fname=input('  Nama folder ['+targets[0].parent.name+']: ').strip() or targets[0].parent.name
 else:fname=None
 ok,links=gofile_upload_files(targets,fname)
 if links:
  msg='<b>Upload Gofile</b>'
  for name,url in links:
   print(ok('  '+name))
   print('  '+url+'\n')
   msg=msg+'\n'+name+'\n'+url
  tg_send(msg)
 else:print(er('  Semua upload gagal.'))
 input('\n  Enter...')
def upload_drive():
 hdr('UPLOAD - Google Drive')
 all_files=[]
 for d in [UPLOAD,OUTPUT,Path('/content/extracts'),Path('/content/downloads')]:
  if d.exists():
   for f in sorted(d.rglob('*')):
    if f.is_file() and f.suffix.lower() in V|A|S:all_files.append((d,f))
 if not all_files:print(er('  Tidak ada file untuk di-upload.'));input('  Enter...');return
 print()
 idx=0
 for d in [UPLOAD,OUTPUT,Path('/content/extracts'),Path('/content/downloads')]:
  grp=[(dd,f) for dd,f in all_files if dd==d]
  if not grp:continue
  print('  ['+d.name+'/]  ('+str(len(grp))+' file)')
  for dd,f in grp:
   size=f.stat().st_size/1024/1024
   print('    ['+str(idx)+'] '+f.name+'  '+dim(str(int(size))+'MB'))
   idx+=1
  print()
 flat=[f for dd,f in all_files]
 c=input('  Pilih (* semua / 0,1,2 / 0-3 / Q batal): ').strip().upper()
 if c=='Q':return
 if c=='*':targets=flat
 else:
  try:
   nums=[]
   for part in c.split(','):
    part=part.strip()
    if '-' in part:a,b=part.split('-',1);nums.extend(range(int(a),int(b)+1))
    else:nums.append(int(part))
   targets=[flat[n] for n in nums if 0<=n<len(flat)]
  except:print('  Input tidak valid.');input('  Enter...');return
  if not targets:return
 cid=gdrive_secret('GDRIVE_CLIENT_ID');sec=gdrive_secret('GDRIVE_CLIENT_SECRET');ref=gdrive_secret('GDRIVE_REFRESH_TOKEN')
 parent_id=gdrive_secret('GDRIVE_FOLDER_ID') or '1pjpd63PTFvwYd8iI7dvMwcU-e_LMqvUE'
 if not(cid and sec and ref):
  print(er('  Secret GDrive tidak kebaca.'));print('  Aktifkan toggle secret + re-run cell Install.');input('  Enter...');return
 print('  Auth via API...')
 tok=gdrive_token(cid,sec,ref)
 if not tok:print(er('  Gagal dapat access token.'));return
 m=re.search(r'/folders/([A-Za-z0-9_-]+)',parent_id)
 if m:parent_id=m.group(1)
 elif len(parent_id)<20:
  q="name='"+parent_id+"' and mimeType='application/vnd.google-apps.folder' and trashed=false"
  try:
   r=requests.get('https://www.googleapis.com/drive/v3/files',headers={'Authorization':'Bearer '+tok},params={'q':q,'fields':'files(id)'},timeout=15)
   fs=r.json().get('files',[])
   if fs:parent_id=fs[0]['id']
  except:pass
 sub=input('  Subfolder ['+dim('langsung ke parent')+']: ').strip()
 target=parent_id
 if sub:
  try:
   q2="name='"+sub+"' and '"+parent_id+"' in parents and mimeType='application/vnd.google-apps.folder' and trashed=false"
   r2=requests.get('https://www.googleapis.com/drive/v3/files',headers={'Authorization':'Bearer '+tok},params={'q':q2,'fields':'files(id)'},timeout=15)
   fs2=r2.json().get('files',[])
   if fs2:target=fs2[0]['id']
   else:
    meta={'name':sub,'mimeType':'application/vnd.google-apps.folder','parents':[parent_id]}
    r3=requests.post('https://www.googleapis.com/drive/v3/files',headers={'Authorization':'Bearer '+tok,'Content-Type':'application/json'},data=json.dumps(meta),timeout=15)
    nid=r3.json().get('id')
    if nid:target=nid;print('  Subfolder dibuat: '+sub)
    else:print(er('  Gagal buat subfolder.'))
  except:print(er('  Error buat subfolder.'))
 ok_n=0;fail=[]
 for f in targets:
  print('  Upload '+f.name+' ('+str(round(f.stat().st_size/1024/1024,1))+'MB)...')
  if gdrive_upload_file(tok,f,target):ok_n+=1;print('  '+ok('ok')+' '+f.name)
  else:fail.append(f.name);print('  '+er('gagal')+' '+f.name)
 if ok_n:tg_send('<b>Upload GDrive</b>\n'+str(ok_n)+' file berhasil')
 if fail:print(er('  Gagal: '+', '.join(fail)))
 input('\n  Enter...')
def tg_owner():
 return get_secret('OWNER_ID')
def tg_token():
 return get_secret('HARU_BOT_TOKEN')
def tg_send(msg):
 oid=tg_owner();tok=tg_token()
 if not oid or not tok:return
 try:requests.post('https://api.telegram.org/bot'+tok+'/sendMessage',json={'chat_id':oid,'text':msg,'parse_mode':'HTML','disable_web_page_preview':True},timeout=10)
 except:pass
def menu_upload():
 while True:
  ci();hdr('UPLOAD')
  print()
  print('  [1] Gofile  (folder gabungan)')
  print('  [2] Google Drive (multi-file + subfolder)')
  print()
  print('  [0] Kembali')
  print()
  c=input('  Pilih: ').strip()
  if c=='0':return
  elif c=='1':upload_gofile()
  elif c=='2':upload_drive()
  input('\n  Enter...')
def main():
 load_secrets()
 menu_upload()
if __name__=='__main__':main()
'''
meta_path = '/usr/local/bin/haru-metadata'
with open(meta_path, 'w') as f:
    f.write(HARU_META_SCRIPT)
os.chmod(meta_path, 0o755)
print('HARU-AUTORENAME installing...')
autorename_path = '/usr/local/bin/auto-rename'
with open(autorename_path, 'w') as f:
    f.write(HARU_AUTORENAME_SCRIPT)
os.chmod(autorename_path, 0o755)
print('HARU-DOWNLOAD installing...')
download_path = '/usr/local/bin/haru-download'
with open(download_path, 'w') as f:
    f.write(HARU_DOWNLOAD_SCRIPT)
os.chmod(download_path, 0o755)
print('HARU-UPLOAD installing...')
upload_path = '/usr/local/bin/haru-upload'
with open(upload_path, 'w') as f:
    f.write(HARU_UPLOAD_SCRIPT)
os.chmod(upload_path, 0o755)
# aliases biar gampang
try:
    with open(os.path.expanduser('~/.bashrc'), 'a') as bf:
        bf.write("\nalias haru-mux='/usr/local/bin/haru-mux'\n")
        bf.write("\nalias haru-metadata='/usr/local/bin/haru-metadata'\n")
        bf.write("\nalias haru-download='/usr/local/bin/haru-download'\n")
        bf.write("\nalias haru-upload='/usr/local/bin/haru-upload'\n")
        bf.write("\nalias auto-rename='/usr/local/bin/auto-rename'\n")
except Exception:
    pass
# Export secrets untuk terminal (userdata tidak terbaca dari ttyd/tmux)
try:
    _secrets = {}
    try:
        from google.colab import userdata as _ud
        for _k in ['GOFILE_API_TOKEN','GDRIVE_CLIENT_ID','GDRIVE_CLIENT_SECRET','GDRIVE_REFRESH_TOKEN','GDRIVE_FOLDER_ID','OWNER_ID','HARU_BOT_TOKEN']:
            try:
                _v = _ud.get(_k)
                if _v: _secrets[_k]=str(_v).strip()
            except Exception: pass
    except Exception: pass
    for _k,_v in _secrets.items(): os.environ[_k]=_v
    if _secrets:
        import json as _js
        _old = {}
        if os.path.exists('/content/.haru_secrets.json'):
            try: _old = _js.load(open('/content/.haru_secrets.json'))
            except Exception: pass
        _old.update(_secrets)
        with open('/content/.haru_secrets.json','w') as _sf: _js.dump(_old,_sf)
        os.chmod('/content/.haru_secrets.json',0o600)
        print('  Secrets untuk terminal: '+', '.join(sorted(_secrets.keys())))
    else:
        print('  (Belum ada secret terbaca - aktifkan di menu Rahasia.)')
except Exception:
    print('  (Skip export secrets.)')

print('\n' + '='*62)
print('✅ haru-mux terinstall!')
print('='*62)
print()
print('  👉 Klik tombol "Terminal" di bagian bawah Colab')
print('     (sebelah kiri tombol "Python 3")')
print()
print('  Lalu ketik:\n')
print('    haru-mux       (muxing series/single)\n')
print('    haru-extract   (extract track)\n')
print('    haru-metadata  (edit metadata)\n')
print('    haru-download  (download file)\n')
print('    haru-upload    (upload file)\n')
print('  Menyiapkan web terminal di bawah...\n')
print('='*62)

print('Setup web terminal...')
try:
    _s2 = {}
    try:
        from google.colab import userdata as _ud
        for _k in ['GOFILE_API_TOKEN','GDRIVE_CLIENT_ID','GDRIVE_CLIENT_SECRET','GDRIVE_REFRESH_TOKEN','GDRIVE_FOLDER_ID','OWNER_ID','HARU_BOT_TOKEN']:
            try:
                _v = _ud.get(_k)
                if _v: _s2[_k]=str(_v).strip()
            except Exception: pass
    except Exception: pass
    if _s2:
        import json as _js
        try: _old = _js.load(open('/content/.haru_secrets.json'))
        except Exception: _old = {}
        _old.update(_s2)
        with open('/content/.haru_secrets.json','w') as _sf: _js.dump(_old,_sf)
        os.chmod('/content/.haru_secrets.json',0o600)
        print('  Secrets refresh: '+', '.join(sorted(_old.keys())))
except Exception: pass
subprocess.run(['apt-get', 'update', '-qq'], capture_output=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'tmux'], capture_output=True)
if not os.path.exists('/usr/local/bin/cloudflared'):
    print('  Download cloudflared...')
    subprocess.run(['curl', '-s', '-L', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-o', '/usr/local/bin/cloudflared'])
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'])
if not os.path.exists('/usr/local/bin/ttyd'):
    print('  Download ttyd...')
    subprocess.run(['curl', '-s', '-L', 'https://github.com/tsl0922/ttyd/releases/latest/download/ttyd.x86_64', '-o', '/usr/local/bin/ttyd'])
    subprocess.run(['chmod', '+x', '/usr/local/bin/ttyd'])

subprocess.run(['pkill', '-f', 'ttyd'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared tunnel'], capture_output=True)
time.sleep(1)

subprocess.run(['tmux', 'set', '-g', 'history-limit', '50000'], capture_output=True)
subprocess.run(['tmux', 'set', '-g', 'mouse', 'on'], capture_output=True)
subprocess.Popen(['/usr/local/bin/ttyd', '-p', '7681', '-W', '-t', 'fontSize=15', 'tmux', 'new-session', '-A', '-s', 'haru', 'bash'], cwd='/content', stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)

print('  Buka tunnel Cloudflare...')
cf = subprocess.Popen(['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7681'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
web_url = None
end = time.time() + 35
while time.time() < end:
    line = cf.stdout.readline()
    if not line:
        time.sleep(0.3)
        continue
    m = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        web_url = m[-1]
        break

print()
print('=' * 62)
if web_url:
    print('WEB TERMINAL SIAP:')
    print('  ' + web_url)
    print()
    print('  Di HP: buka link di atas (keyboard: pakai Hacker Keyboard / terminal fullscreen).')
    print('  Di laptop: tinggal buka, copy-paste & arrow keys jalan normal.')
    print('  Perintah: haru-mux  |  haru-extract')
    try:
        from google.colab import userdata as _ud
        _oid = _ud.get('OWNER_ID') or ''
    except Exception:
        _oid = os.environ.get('OWNER_ID') or ''
    try:
        from google.colab import userdata as _ud2
        _tg = _ud2.get('HARU_BOT_TOKEN') or ''
    except Exception:
        _tg = os.environ.get('HARU_BOT_TOKEN') or ''
    if _oid and _tg:
        try:
            requests.post('https://api.telegram.org/bot' + _tg + '/sendMessage', json={'chat_id': _oid, 'text': '<b>HaruColab terminal siap!</b>\nWeb: ' + web_url + '\nKetik: haru-mux / haru-extract', 'parse_mode': 'HTML', 'disable_web_page_preview': True}, timeout=8)
            print('  Notif Telegram terkirim.')
        except Exception as _e:
            print('  Gagal kirim Telegram: ' + str(_e)[:100])
    else:
        print('  (Aktifkan HARU_BOT_TOKEN & OWNER_ID di Secrets biar link auto-post ke Telegram.)')
    display(HTML('<a href="' + web_url + '" target="_blank" style="background:#238636;color:#fff;padding:12px 24px;text-decoration:none;border-radius:6px;font-weight:bold;display:inline-block;">Buka Web Terminal</a>'))
else:
    print('Gagal dapat URL tunnel. Jalankan ulang cell ini.')
print('=' * 62)
print()
print('Biarkan cell ini running agar tunnel tetap hidup.')
try:
    while True:
        time.sleep(30)
except KeyboardInterrupt:
    print('Web terminal ditutup.')


## 1C — Terminal di dalam Cell (colab-xterm)
Enak di HP & bisa fullscreen. Jalankan cell di bawah, terminal muncul di dalam cell — ketik `haru-mux` di sana.

In [ ]:
#@title Buka Terminal di Cell { display-mode: "form" }
!pip install colab-xterm -q
%load_ext colabxterm
%xterm


---
## Jalur alternatif — form per cell
Bagian bawah ini versi form satu-per-satu (alternatif web terminal di atas). Boleh diskip kalau sudah pakai `haru-mux` / `haru-extract`.

In [ ]:
#@title Gofile Downloader { display-mode: "form" }
#@markdown ### Pilih mode download
mode = "Folder (auto-detect semua file)" #@param ["Satu file", "Folder (auto-detect semua file)"]

#@markdown ---
#@markdown ### Isi link Gofile
gofile_url = "" #@param {type:"string"}
gofile_password = "" #@param {type:"string"}

#@markdown ---
#@markdown ### (Opsional) Nama file override — kosongkan untuk auto
gofile_filename = "" #@param {type:"string"}


GOFILE_PROXY_API = 'https://go.filmbeehub.workers.dev/api/v1/generate'
GOFILE_PROXY_DATA = 'https://go.filmbeehub.workers.dev/api/data'


def get_gofile_token():
    try:
        from google.colab import userdata
        return userdata.get('GOFILE_API_TOKEN')
    except Exception:
        pass
    return os.environ.get('GOFILE_API_TOKEN')


def gofile_generate_link(url, token, password=''):
    """Generate direct download link via filmbeehub proxy."""
    payload = {'url': url, 'password': password, 'expiresInSeconds': 3600, 'filePage': 0, 'fileSize': 100}
    headers = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}
    resp = requests.post(GOFILE_PROXY_API, json=payload, headers=headers, timeout=60)
    return resp.json()


def gofile_get_folder_files(url, token, password=''):
    """
    Ambil semua file dari folder Gofile via filmbeehub proxy.
    Flow: generate link -> dapat share ID -> fetch /api/data/{shareId}
    Return list: [{'name': ..., 'size': ..., 'bytes': ..., 'downloadUrl': ...}]
    """
    # Step 1: Generate link via proxy
    result = gofile_generate_link(url, token, password)
    if not result.get('ok'):
        print(f'  ❌ Gagal generate: {result.get("error", "unknown")}')
        return []

    data = result.get('data', {})

    # Step 2a: Jika langsung dapat downloadLinks (single file / small folder)
    if data.get('downloadLinks'):
        return data['downloadLinks']

    # Step 2b: Jika ada shareUrl, fetch dari /api/data/{shareId}
    share_url = data.get('shareUrl', '')
    if share_url:
        share_id = share_url.rstrip('/').split('/')[-1]
        print(f'  🔗 Share ID: {share_id}')
        resp = requests.get(f'{GOFILE_PROXY_DATA}/{share_id}',
                            headers={'User-Agent': 'Mozilla/5.0'}, timeout=30)
        folder_data = resp.json()
        all_files = []
        for group in folder_data.get('groups', []):
            all_files.extend(group.get('files', []))
        return all_files

    return []


def detect_type(filepath):
    ext = filepath.suffix.lower()
    video_exts = {'.mkv','.mp4','.avi','.mov','.webm','.flv','.wmv','.ts','.m4v'}
    audio_exts = {'.mp3','.aac','.flac','.wav','.ogg','.opus','.mka','.ac3','.dts','.eac3','.m4a'}
    sub_exts   = {'.srt','.ass','.ssa','.sub','.idx','.sup','.vtt','.pgs','.scc','.sami'}
    if ext in video_exts: return 'video'
    if ext in audio_exts: return 'audio'
    if ext in sub_exts:   return 'subtitle'
    return 'other'


def gofile_download_file(url, password='', token=None, fname_override=None):
    if not token: token = get_gofile_token()
    if not token:
        print('  ❌ GOFILE_API_TOKEN tidak ditemukan di Colab Secrets.')
        return None
    print('  🔗 Request direct link...')
    result = gofile_generate_link(url, token, password)
    if not result.get('ok'):
        print(f'  ❌ Gagal: {result.get("error", "unknown")}')
        return None
    data = result.get('data', {})
    # Single file: langsung ada downloadLinks
    links = data.get('downloadLinks', [])
    if not links:
        print(f'  ❌ Tidak ada download link.')
        return None
    link = links[0]
    direct_url = link['downloadUrl']
    fname = fname_override or link.get('name', '')
    print(f'  📥 Downloading {fname}...')
    resp = requests.get(direct_url, stream=True, timeout=600)
    resp.raise_for_status()
    if not fname:
        cd = resp.headers.get('Content-Disposition', '')
        m = re.search(r'filename[*]?=["\']?([^"\';\n]+)', cd)
        fname = urllib.parse.unquote(m.group(1).strip()) if m else hashlib.md5(url.encode()).hexdigest()[:12]
    dest = UPLOAD_DIR / fname
    total = 0
    with open(dest, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=1024*1024):
            f.write(chunk)
            total += len(chunk)
    print(f'  ✅ {fname}  ({total:,} bytes / {total/1024/1024:.1f} MB)')
    return dest


def gofile_download_folder(url, password='', token=None):
    if not token: token = get_gofile_token()
    if not token:
        print('  ❌ GOFILE_API_TOKEN tidak ditemukan di Colab Secrets.')
        return []
    print('  🔍 Ambil daftar file di folder...')
    folder_files = gofile_get_folder_files(url, token, password)
    if not folder_files:
        print('  ❌ Folder kosong atau tidak bisa diakses.')
        return []
    print(f'  📋 Ditemukan {len(folder_files)} file:')
    for f in folder_files:
        ft = detect_type(Path(f['name']))
        size_str = f.get('size', '?')
        print(f'     [{ft:<9}] {f["name"]}  ({size_str})')
    print()
    downloaded = []
    for i, f in enumerate(folder_files, 1):
        print(f'  [{i}/{len(folder_files)}] {f["name"]}')
        try:
            dl_url = f.get('downloadUrl', '')
            if not dl_url:
                print(f'    ⚠️  Tidak ada download URL, skip.')
                continue
            resp = requests.get(dl_url, stream=True, timeout=600)
            resp.raise_for_status()
            dest = UPLOAD_DIR / f['name']
            total = 0
            with open(dest, 'wb') as fh:
                for chunk in resp.iter_content(chunk_size=1024*1024):
                    fh.write(chunk)
                    total += len(chunk)
            ft = detect_type(dest)
            print(f'    ✅ [{ft:<9}] {f["name"]}  ({total:,} bytes)')
            downloaded.append(dest)
        except Exception as e:
            print(f'    ❌ Gagal: {e}')
        print()
    return downloaded


# ─── Jalankan ───
if gofile_url.strip():
    if mode.startswith('Folder'):
        gofile_download_folder(gofile_url.strip(), gofile_password)
    else:
        gofile_download_file(gofile_url.strip(), gofile_password, fname_override=gofile_filename or None)
else:
    print('⏭️  Isi gofile_url di form sebelah kanan, lalu jalankan ulang.')

## 3 — Download dari Google Drive

In [ ]:
#@title Google Drive Downloader { display-mode: "form" }
#@markdown ### Path file/folder di Google Drive (setelah mount)
#@markdown Contoh: `/content/drive/MyDrive/Movies/film.mkv`
gdrive_path = "" #@param {type:"string"}

#@markdown ---
#@markdown ### (Opsional) Nama file override — kosongkan untuk auto
gdrive_filename = "" #@param {type:"string"}


def mount_drive():
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        print('✅ Google Drive mounted.')
    except Exception as e:
        print(f'❌ Gagal mount: {e}')


if gdrive_path.strip():
    src = Path(gdrive_path.strip())
    if not src.exists():
        mount_drive()
    if src.exists():
        if src.is_dir():
            print(f'📂 Copy semua file dari folder: {src}\n')
            for f in src.iterdir():
                if f.is_file():
                    dest_name = gdrive_filename.strip() if gdrive_filename.strip() else f.name
                    shutil.copy2(f, UPLOAD_DIR / dest_name)
                    print(f'  ✅ {f.name}  →  {dest_name}')
        else:
            dest_name = gdrive_filename.strip() if gdrive_filename.strip() else src.name
            shutil.copy2(src, UPLOAD_DIR / dest_name)
            print(f'✅ {src.name}  →  {dest_name}')
    else:
        print(f'❌ Tidak ditemukan: {gdrive_path}')
else:
    print('⏭️  Isi gdrive_path di form sebelah kanan, lalu jalankan ulang.')

## 4 — Download dari Direct URL

In [ ]:
#@title Direct URL Downloader { display-mode: "form" }
#@markdown ### URL file
direct_url = "" #@param {type:"string"}

#@markdown ---
#@markdown ### (Opsional) Nama file override — kosongkan untuk auto
direct_filename = "" #@param {type:"string"}


if direct_url.strip():
    print(f'📥 Download dari URL...')
    try:
        resp = requests.get(direct_url.strip(), stream=True, timeout=300, allow_redirects=True)
        resp.raise_for_status()
        if direct_filename.strip():
            fname = direct_filename.strip()
        else:
            cd = resp.headers.get('Content-Disposition', '')
            m = re.search(r'filename[*]?=["\']?([^"\';\n]+)', cd)
            if m:
                fname = urllib.parse.unquote(m.group(1).strip())
            else:
                parsed = urllib.parse.urlparse(direct_url.strip())
                fname = Path(parsed.path).name or hashlib.md5(direct_url.encode()).hexdigest()[:12]
        dest = UPLOAD_DIR / fname
        total = 0
        with open(dest, 'wb') as f:
            for chunk in resp.iter_content(chunk_size=1024*1024):
                f.write(chunk)
                total += len(chunk)
        print(f'  ✅ {fname}  ({total:,} bytes / {total/1024/1024:.1f} MB)')
    except Exception as e:
        print(f'  ❌ Error: {e}')
else:
    print('⏭️  Isi direct_url di form sebelah kanan, lalu jalankan ulang.')

## 5 — Upload Manual

In [ ]:
#@title Upload File dari PC { display-mode: "form" }
#@markdown Jalankan cell ini untuk upload file langsung dari komputer.
try:
    from google.colab import files
    print('📤 Upload file (video/audio/subtitle):')
    uploaded = files.upload()
    for name, data in uploaded.items():
        dest = UPLOAD_DIR / name
        with open(dest, 'wb') as f:
            f.write(data)
        print(f'  ✅ {name}  ({len(data):,} bytes)')
except ImportError:
    print('⚠️  Bukan di Colab — skip upload.')

## 6 — Lihat File & Register Track

In [ ]:
#@title Lihat Semua File { display-mode: "form" }
#@markdown Klik **Run** untuk melihat file yang sudah terkumpul di `/content/uploads/`
files_list = sorted(UPLOAD_DIR.iterdir())
if files_list:
    print(f'📂 {len(files_list)} file di /content/uploads/:\n')
    for f in files_list:
        size = f.stat().st_size
        ft = detect_type(f)
        print(f'  [{ft:<9}] {f.name:<45} {size:>12,} bytes  ({size/1024/1024:.1f} MB)')
else:
    print('📂 Belum ada file. Jalankan cell download/upload di atas dulu.')

In [ ]:
#@title Register Semua Track { display-mode: "form" }
#@markdown Jalankan untuk scan semua file dan register sebagai track.
TRACK_ID_COUNTER = 0

def probe_file(filepath):
    rj = subprocess.run(
        ['mkvmerge', '-J', str(filepath)],
        capture_output=True, text=True, timeout=30
    )
    tracks_json = []
    if rj.returncode == 0 and rj.stdout.strip():
        try:
            dj = json.loads(rj.stdout)
            for tr in dj.get('tracks', []):
                pr = tr.get('properties', {}) or {}
                tracks_json.append({'mkvmerge_id': tr.get('id', 0), 'codec': str(tr.get('codec', '')), 'type': str(tr.get('type', '')).lower(), 'language': str(pr.get('language', 'und')).lower(), 'track_name': str(pr.get('track_name', '') or ''), 'default_track': 'yes' if pr.get('default_track', False) else 'no'})
        except Exception:
            pass
    result = subprocess.run(
        ['mkvmerge', '--identify-verbose', str(filepath)],
        capture_output=True, text=True, timeout=30
    )
    return {'tracks_json': tracks_json, 'stdout': result.stdout, 'stderr': result.stderr, 'returncode': result.returncode}


def parse_tracks_from_probe(probe):
    tracks = []
    # Cek stdout DAN stderr (mkvmerge kadang output ke stderr)
    for text in [probe['stdout'], probe['stderr']]:
        for line in text.splitlines():
            m = re.match(r'\s*Track ID (\d+): (.+?)\s+\((.+?)\)', line)
            if m:
                tid = int(m.group(1))
                # Hindari duplikat
                if not any(t['mkvmerge_id'] == tid for t in tracks):
                    tracks.append({'mkvmerge_id': tid, 'codec': m.group(2).strip(), 'type': m.group(3).strip().lower()})
    return tracks


def register_file(filepath):
    global TRACK_ID_COUNTER
    entries = []
    file_type = detect_type(filepath)
    probe = probe_file(filepath)
    detected = parse_tracks_from_probe(probe)
    if not detected:
        detected = [{'mkvmerge_id': 0, 'codec': file_type, 'type': file_type}]
    for d in detected:
        t_raw = d['type']
        if t_raw == 'subtitles': t_raw = 'subtitle'
        track_type = t_raw if t_raw in ('video','audio','subtitle') else file_type
        entry = {
            'local_id': TRACK_ID_COUNTER,
            'source_file': str(filepath),
            'source_name': filepath.name,
            'mkvmerge_track_id': d['mkvmerge_id'],
            'codec': d['codec'],
            'type': track_type,
            'language': d.get('language', 'und'),
            'track_name': d.get('track_name', ''),
            'default_track': d.get('default_track', 'no'),
            'forced': 'no',
            'hearing_impaired': 'no',
            'visual_impaired': 'no',
            'commentary': 'no',
            'original': 'no',
            'delay': 0,
            'copy': 'no',
            'enabled': True,
        }
        TRACK_ID_COUNTER += 1
        entries.append(entry)
    return entries


all_tracks = []
for fp in sorted(UPLOAD_DIR.iterdir()):
    if fp.is_file():
        print(f'🔍 {fp.name}')
        entries = register_file(fp)
        for e in entries:
            print(f'   → Track {e["local_id"]}: {e["type"]} — {e["codec"]}')
        all_tracks.extend(entries)

print(f'\n📋 Total {len(all_tracks)} track terdaftar.')

## 7 — Lihat & Edit Track

In [ ]:
#@title Lihat Semua Track { display-mode: "form" }
def print_tracks():
    if not all_tracks:
        print('(kosong)')
        return
    print(f'{"ID":<4} {"Type":<10} {"Codec":<20} {"Source":<30} {"Lang":<5} {"Name":<20} {"Default":<8} {"Forced":<7} {"Delay":<10} {"En":<4}')
    print('─' * 130)
    for t in all_tracks:
        en = '✅' if t['enabled'] else '❌'
        print(f'{t["local_id"]:<4} {t["type"]:<10} {t["codec"]:<20} {t["source_name"]:<30} {t["language"]:<5} {t["track_name"]:<20} {t["default_track"]:<8} {t["forced"]:<7} {t["delay"]:>8}ms {en}')
print_tracks()

In [ ]:
#@title Edit Track { display-mode: "form" }
#@markdown ### Pilih track yang mau diedit
track_id = 0 #@param {type:"integer"}

#@markdown ### Bahasa (ISO 639-1)
language = "und" #@param ["und", "id", "en", "ja", "ko", "zh", "ms", "ar", "de", "fr", "es", "pt", "ru", "it", "th", "vi", "hi", "tr", "pl", "nl"]

#@markdown ### Nama Track
track_name = "" #@param {type:"string"}

#@markdown ### Default Track
default_track = "no" #@param ["yes", "no"]

#@markdown ### Forced
forced = "no" #@param ["yes", "no"]

#@markdown ### Delay (ms, positif=tunda, negatif=maju)
delay = 0 #@param {type:"integer"}

#@markdown ### Flags lainnya
hearing_impaired = "no" #@param ["yes", "no"]
visual_impaired = "no" #@param ["yes", "no"]
commentary = "no" #@param ["yes", "no"]
original = "no" #@param ["yes", "no"]

#@markdown ### Aktifkan track ini?
enabled = True #@param {type:"boolean"}


found = False
for t in all_tracks:
    if t['local_id'] == track_id:
        t['language'] = language
        if track_name.strip(): t['track_name'] = track_name.strip()
        t['default_track'] = default_track
        t['forced'] = forced
        t['delay'] = delay
        t['hearing_impaired'] = hearing_impaired
        t['visual_impaired'] = visual_impaired
        t['commentary'] = commentary
        t['original'] = original
        t['enabled'] = enabled
        found = True
        break

if found:
    print(f'✅ Track {track_id} updated.')
    print_tracks()
else:
    print(f'❌ Track {track_id} tidak ditemukan.')

In [ ]:
#@title Batch Edit — Terapkan ke Banyak Track { display-mode: "form" }
#@markdown ### Pilih track yang mau diedit
target_type = "Semua" #@param ["Semua", "Video", "Audio", "Subtitle"]
target_ids = "" #@param {type:"string"}

#@markdown ---
#@markdown ### Yang mau diubah (kosongkan jika tidak diubah)
batch_language = "" #@param ["", "id", "en", "ja", "ko", "zh", "ms", "ar", "de", "fr", "es", "pt", "ru", "it", "th", "vi", "hi", "tr", "pl", "nl"]
batch_track_name = "" #@param {type:"string"}
batch_default = "" #@param ["", "yes", "no"]
batch_forced = "" #@param ["", "yes", "no"]
batch_delay = 0 #@param {type:"integer"}

#@markdown ---
#@markdown ### Flags (isi `yes` atau `kosongkan`)
batch_hearing_impaired = "" #@param ["", "yes", "no"]
batch_visual_impaired = "" #@param ["", "yes", "no"]
batch_commentary = "" #@param ["", "yes", "no"]
batch_original = "" #@param ["", "yes", "no"]

#@markdown ---
#@markdown ### ✅ Centang untuk apply
apply_batch = False #@param {type:"boolean"}


if not apply_batch:
    print('ℹ️  Centang apply_batch dulu, lalu jalankan ulang.')
else:
    # Parse target IDs
    selected_ids = set()
    if target_ids.strip():
        for part in target_ids.split(','):
            part = part.strip()
            if '-' in part:
                start, end = part.split('-', 1)
                selected_ids.update(range(int(start), int(end) + 1))
            elif part.isdigit():
                selected_ids.add(int(part))

    # Map type
    type_map = {'Semua': None, 'Video': 'video', 'Audio': 'audio', 'Subtitle': 'subtitle'}
    target_t = type_map[target_type]

    count = 0
    for t in all_tracks:
        # Filter by type
        if target_t and t['type'] != target_t:
            continue
        # Filter by IDs (if specified)
        if selected_ids and t['local_id'] not in selected_ids:
            continue

        # Apply changes
        if batch_language:          t['language'] = batch_language
        if batch_track_name.strip(): t['track_name'] = batch_track_name.strip()
        if batch_default:            t['default_track'] = batch_default
        if batch_forced:             t['forced'] = batch_forced
        if batch_delay != 0:         t['delay'] = batch_delay
        if batch_hearing_impaired:   t['hearing_impaired'] = batch_hearing_impaired
        if batch_visual_impaired:    t['visual_impaired'] = batch_visual_impaired
        if batch_commentary:         t['commentary'] = batch_commentary
        if batch_original:           t['original'] = batch_original
        count += 1

    print(f'✅ Batch edit: {count} track diupdate.\n')
    print_tracks()

In [ ]:
#@title Atur Default Track { display-mode: "form" }
#@markdown ### Pilih track yang mau dijadikan default
#@markdown Jalankan cell "Lihat Semua Track" dulu untuk melihat ID.
set_default_id = -1 #@param {type:"integer"}

#@markdown ### Atau: reset semua default ke "No" dulu
clear_all_defaults = False #@param {type:"boolean"}


if clear_all_defaults:
    for t in all_tracks:
        t['default_track'] = 'no'
    print('🔄 Semua default track direset ke "no".\n')

if set_default_id >= 0:
    found = False
    for t in all_tracks:
        if t['local_id'] == set_default_id:
            target_type = t['type']
            # Clear default lain yang se-tipe
            cleared = 0
            for other in all_tracks:
                if other['type'] == target_type and other['default_track'] == 'yes':
                    other['default_track'] = 'no'
                    cleared += 1
            t['default_track'] = 'yes'
            print(f'✅ Track {set_default_id} ({t["source_name"]}) dijadikan default {target_type}.')
            if cleared:
                print(f'   🔄 {cleared} track {target_type} lain direset ke "no".')
            found = True
            break
    if not found:
        print(f'❌ Track {set_default_id} tidak ditemukan.')

if set_default_id < 0 and not clear_all_defaults:
    print('ℹ️  Isi set_default_id atau centang clear_all_defaults, lalu jalankan ulang.')

print()
print_tracks()

In [ ]:
#@title Tambah / Hapus Track { display-mode: "form" }
#@markdown ### Duplikat track
dup_track_id = -1 #@param {type:"integer"}

#@markdown ### Hapus track
del_track_id = -1 #@param {type:"integer"}

if dup_track_id >= 0:
    for t in all_tracks:
        if t['local_id'] == dup_track_id:
            new_t = dict(t)
            new_t['local_id'] = TRACK_ID_COUNTER
            TRACK_ID_COUNTER += 1
            all_tracks.append(new_t)
            print(f'✅ Track {dup_track_id} diduplikasi → ID baru {new_t["local_id"]}')
            break
    else:
        print(f'❌ Track {dup_track_id} tidak ditemukan.')

if del_track_id >= 0:
    before = len(all_tracks)
    all_tracks = [t for t in all_tracks if t['local_id'] != del_track_id]
    if len(all_tracks) < before:
        print(f'🗑️  Track {del_track_id} dihapus.')
    else:
        print(f'❌ Track {del_track_id} tidak ditemukan.')

if dup_track_id < 0 and del_track_id < 0:
    print('ℹ️  Isi dup_track_id atau del_track_id di form, lalu jalankan ulang.')

print()
print_tracks()

## 8 — Mux

In [ ]:
#@title Konfigurasi Output { display-mode: "form" }
#@markdown ### Nama file output (kosongkan = otomatis dari nama video)
output_filename = "" #@param {type:"string"}

# Auto-detect dari file video pertama
if not output_filename.strip():
    video_tracks = [t for t in all_tracks if t['type'] == 'video']
    if video_tracks:
        video_stem = Path(video_tracks[0]['source_name']).stem
        output_filename = video_stem + '.mkv'
    else:
        output_filename = 'output.mkv'

OUTPUT_PATH = OUTPUT_DIR / output_filename
print(f'📁 Output: {OUTPUT_PATH}')

In [ ]:
#@title Mux Sekarang { display-mode: "form" }
#@markdown ### Auto-fix default track? (recommended)
#@markdown Satu tipe = satu default. Jika ada lebih dari 1, yang pertama dipertahankan.
auto_fix_default = True #@param {type:"boolean"}

def enforce_single_default_per_type():
    """Pastikan per tipe (video/audio/subtitle) cuma ada 1 default track."""
    fixed = 0
    for track_type in ['video', 'audio', 'subtitle']:
        defaults = [t for t in all_tracks if t['type'] == track_type and t['default_track'] == 'yes']
        if len(defaults) > 1:
            for t in defaults[1:]:
                t['default_track'] = 'no'
                fixed += 1
        elif len(defaults) == 0:
            # Belum ada default → set yang pertama
            first = next((t for t in all_tracks if t['type'] == track_type), None)
            if first:
                first['default_track'] = 'yes'
                fixed += 1
    return fixed

def build_mux_command():
    by_file = {}
    for t in all_tracks:
        if not t['enabled']:
            continue
        by_file.setdefault(t['source_file'], []).append(t)
    cmd = ['mkvmerge', '-o', str(OUTPUT_PATH)]
    for filepath, tracks in by_file.items():
        cmd.extend(['--no-chapters', '--no-global-tags'])
        for t in tracks:
            tid = str(t['mkvmerge_track_id'])
            if t['track_name']:
                cmd.extend(['--track-name', f'{tid}:{t["track_name"]}'])
            if t['language'] and t['language'] != 'und':
                cmd.extend(['--language', f'{tid}:{t["language"]}'])
            if t['default_track'] != 'auto':
                cmd.extend(['--default-track', f'{tid}:{t["default_track"]}'])
            if t['forced'] == 'yes':
                cmd.extend(['--forced-track', f'{tid}:yes'])
            if t['hearing_impaired'] == 'yes':
                cmd.extend(['--hearing-impaired-flag', f'{tid}:yes'])
            if t['visual_impaired'] == 'yes':
                cmd.extend(['--visual-impaired-flag', f'{tid}:yes'])
            if t['commentary'] == 'yes':
                cmd.extend(['--commentary-flag', f'{tid}:yes'])
            if t['original'] == 'yes':
                cmd.extend(['--original-flag', f'{tid}:yes'])
            if t['delay'] != 0:
                cmd.extend(['--sync', f'{tid}:{t["delay"]:+d}'])
        cmd.append(filepath)
    return cmd

if auto_fix_default:
    fixed = enforce_single_default_per_type()
    if fixed:
        print(f'🔧 Auto-fix: {fixed} default track direset (hanya 1 per tipe)\n')

cmd = build_mux_command()
print('🚀 Mulai muxing...\n')
result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)

# Cek apakah output file berhasil dibuat (warning ≠ error)
mux_success = OUTPUT_PATH.exists() and OUTPUT_PATH.stat().st_size > 0

if mux_success:
    file_size = OUTPUT_PATH.stat().st_size
    print(f'✅ Muxing berhasil!')
    print(f'   📄 {OUTPUT_PATH.name}  ({file_size:,} bytes / {file_size/1024/1024:.1f} MB)')
    # Tampilkan warning jika ada (bukan error)
    warnings = [l for l in result.stdout.splitlines() if 'Warning' in l]
    if warnings:
        print(f'\n⚠️  {len(warnings)} warning(s):')
        for w in warnings[:3]:
            print(f'   {w[:100]}')
else:
    print(f'❌ Muxing gagal!')
    print('STDOUT:', result.stdout[-500:] if result.stdout else '')
    print('STDERR:', result.stderr[-500:] if result.stderr else '')

## 8B — MediaInfo (cek hasil)

In [ ]:
#@title Cek MediaInfo { display-mode: "form" }
#@markdown ### Path file (otomatis = hasil muxing terakhir)
mediainfo_path = "" #@param {type:"string"}

#@markdown ### Format output
mediainfo_format = "Text" #@param ["Text", "JSON"]


def get_mediainfo(filepath, fmt='text'):
    """Jalankan mediainfo dan return output."""
    cmd = ['mediainfo']
    if fmt == 'json':
        cmd.append('--Output=JSON')
    cmd.append(str(filepath))
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
    return result.stdout


def parse_mediainfo_tracks(info_text):
    """Parse mediainfo text output jadi list track info."""
    tracks = []
    current_type = None
    current_data = {}
    section_headers = {'General', 'Video', 'Audio', 'Text', 'Menu', 'Image'}
    for line in info_text.splitlines():
        stripped = line.strip()
        if not stripped:
            continue
        first_word = stripped.split()[0] if stripped.split() else ''
        # Handle 'Audio #1', 'Text #2' etc.
        is_header = first_word in section_headers and (':' not in stripped or stripped.startswith(first_word))
        if is_header:
            if current_type and current_data:
                tracks.append(current_data)
            current_type = stripped
            current_data = {'type': stripped}
            continue
        if ':' in stripped and current_type:
            key, val = stripped.split(':', 1)
            key, val = key.strip(), val.strip()
            if key and val:
                current_data[key] = val
    if current_type and current_data:
        tracks.append(current_data)
    return tracks


# Resolve path
if mediainfo_path.strip():
    mi_path = Path(mediainfo_path.strip())
else:
    mi_path = OUTPUT_PATH

if not mi_path.exists():
    print(f'❌ File tidak ditemukan: {mi_path}')
else:
    print(f'📋 MediaInfo: {mi_path.name}\n')
    fmt = 'json' if mediainfo_format == 'JSON' else 'text'
    info = get_mediainfo(mi_path, fmt)

    if fmt == 'json':
        data = json.loads(info)
        general = data.get('media', {}).get('track', [{}])[0]
        print(f'Format: {general.get("Format", "?")}')
        print(f'Size: {general.get("FileSize", "?")} bytes')
        print(f'Duration: {general.get("Duration", "?")}s')
        print(f'Bitrate: {general.get("OverallBitRate", "?")} bps')
        print()
        for t in data.get('media', {}).get('track', [])[1:]:
            ttype = t.get('Track type', '?')
            codec = t.get('Format', t.get('CodecID', '?'))
            lang = t.get('Language', '-')
            name = t.get('Title', '-')
            default = t.get('Default', '-')
            forced = t.get('Forced', '-')
            print(f'  [{ttype:<11}] {codec:<25} Lang:{lang:<6} Name:{name:<20} Default:{default}  Forced:{forced}')
    else:
        tracks = parse_mediainfo_tracks(info)
        for t in tracks:
            ttype = t.get('Type', '?')
            codec = t.get('Format', t.get('CodecID', '?'))
            lang = t.get('Language', '-')
            name = t.get('Title', '-')
            default = t.get('Default', '-')
            forced = t.get('Forced', '-')
            print(f'  [{ttype:<11}] {codec:<25} Lang:{lang:<6} Name:{name:<20} Default:{default}  Forced:{forced}')

## 9 — Upload Hasil

In [ ]:
#@title Upload ke Gofile (Guest) { display-mode: "form" }
#@markdown ### File yang mau di-upload (path lengkap)
#@markdown Kosongkan untuk upload hasil muxing terakhir.
upload_file_path = "" #@param {type:"string"}


def gofile_get_server():
    resp = requests.get('https://api.gofile.io/servers', timeout=15)
    data = resp.json()
    if data.get('status') == 'ok':
        return data['data']['servers'][0]['name']
    return 'store1'


def gofile_upload(filepath):
    if not filepath.exists():
        print(f'  ❌ File tidak ditemukan: {filepath}')
        return None
    server = gofile_get_server()
    upload_url = f'https://{server}.gofile.io/uploadfile'
    size_mb = filepath.stat().st_size / 1024 / 1024
    print(f'  📤 Upload ke {server}.gofile.io ... ({filepath.name}, {size_mb:.1f} MB)')
    try:
        with open(filepath, 'rb') as f:
            resp = requests.post(upload_url, files={'file': (filepath.name, f)}, timeout=600)
        result = resp.json()
        if result.get('status') == 'ok':
            d = result['data']
            print(f'  ✅ Upload berhasil!')
            print(f'     Download: {d["downloadPage"]}')
            print(f'     Code: {d["code"]}')
            return d
        else:
            print(f'  ❌ Upload gagal: {json.dumps(result, indent=2)}')
            return None
    except Exception as e:
        print(f'  ❌ Error: {e}')
        return None


fp = Path(upload_file_path.strip()) if upload_file_path.strip() else OUTPUT_PATH
print(f'📤 Upload ke Gofile:\n')
result = gofile_upload(fp)
if result:
    print(f'\n📋 Link: {result["downloadPage"]}')

In [ ]:
#@title Upload ke Google Drive { display-mode: "form" }
#@markdown ### Folder tujuan di MyDrive
gdrive_upload_folder = "HaruColab" #@param {type:"string"}

#@markdown ### File yang mau di-upload (path lengkap, kosongkan untuk hasil muxing)
gdrive_upload_file = "" #@param {type:"string"}


def ensure_drive_mounted():
    if Path('/content/drive').exists() and any(Path('/content/drive').iterdir()):
        return True
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        return True
    except Exception as e:
        print(f'❌ Gagal mount Drive: {e}')
        return False


fp = Path(gdrive_upload_file.strip()) if gdrive_upload_file.strip() else OUTPUT_PATH
if ensure_drive_mounted():
    dest_dir = Path(f'/content/drive/MyDrive/{gdrive_upload_folder.strip()}')
    dest_dir.mkdir(parents=True, exist_ok=True)
    if fp.exists():
        shutil.copy2(fp, dest_dir / fp.name)
        print(f'✅ {fp.name}  →  /content/drive/MyDrive/{gdrive_upload_folder.strip()}/')
    else:
        print(f'❌ File tidak ditemukan: {fp}')
else:
    print('❌ Tidak bisa mount Google Drive.')

## 10 — Download Hasil ke PC

In [ ]:
#@title Download ke PC { display-mode: "form" }
try:
    from google.colab import files
    if OUTPUT_PATH.exists():
        files.download(str(OUTPUT_PATH))
    else:
        print('❌ File output tidak ditemukan.')
except ImportError:
    print(f'📂 File ada di: {OUTPUT_PATH}')